In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("PYTORCH_CUDA_ALLOC_CONF set.")

PYTORCH_CUDA_ALLOC_CONF set.


In [2]:
import sys
import subprocess

packages = [
    "langchain==1.3.9",
    "langchain-core==1.4.7",
    "langchain-community==0.4.2",
    "langchain-huggingface==1.2.2",
    "langchain-chroma==1.1.0",
    "langchain-text-splitters==1.1.2",
    "sentence-transformers==3.0.1",
    "chromadb",
    "pymupdf",
    "pdfplumber",
    "rank-bm25",
    "bitsandbytes",
    "accelerate",
    "scikit-learn",
    "bert-score",
]

subprocess.run(["pip", "install", "-q", "--no-cache-dir"] + packages, check=True)

#Auto-restart kernel so all installs are visible immediately

#print("✅ Packages installed — restarting kernel...")

#import IPython

#IPython.Application.instance().kernel.do_shutdown(restart=True)

CompletedProcess(args=['pip', 'install', '-q', '--no-cache-dir', 'langchain==1.3.9', 'langchain-core==1.4.7', 'langchain-community==0.4.2', 'langchain-huggingface==1.2.2', 'langchain-chroma==1.1.0', 'langchain-text-splitters==1.1.2', 'sentence-transformers==3.0.1', 'chromadb', 'pymupdf', 'pdfplumber', 'rank-bm25', 'bitsandbytes', 'accelerate', 'scikit-learn', 'bert-score'], returncode=0)

In [3]:
from pathlib import Path
from langchain_core.documents import Document
import fitz        
import pdfplumber
import numpy as np  
pdf_dir = "/kaggle/input/datasets/shivammusk/sec-filings/SEC Filings"
pdf_files = list(Path(pdf_dir).glob("*.pdf"))

print(f"Found {len(pdf_files)} PDF files\n")

all_documents = []

for pdf_path in pdf_files:
    print(f"Processing: {pdf_path.name}")
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text").strip()

        if len(text) < 30:
            continue

        # Text block
        all_documents.append(Document(
            page_content=text,
            metadata={
                "source": str(pdf_path),
                "file_name": pdf_path.name,
                "element_type": "Text",
                "page_number": page_num + 1,
            }
        ))

        # Table extraction
        try:
            with pdfplumber.open(pdf_path) as pdf:
                plumber_page = pdf.pages[page_num]
                tables = plumber_page.extract_tables()
                for idx, table in enumerate(tables):
                    if table and len(table) > 1:
                        table_text = "\n".join(
                            [" | ".join(str(cell) if cell is not None else "" for cell in row)
                             for row in table]
                        )
                        all_documents.append(Document(
                            page_content=table_text,
                            metadata={
                                "source": str(pdf_path),
                                "file_name": pdf_path.name,
                                "element_type": "Table",
                                "page_number": page_num + 1,
                                "table_index": idx,
                            }
                        ))
        except Exception:
            continue

    doc.close()

print(f"\n✅ Extraction complete!")
print(f"Total Documents : {len(all_documents)}")
print(f"Text Blocks     : {sum(1 for d in all_documents if d.metadata['element_type'] == 'Text')}")
print(f"Tables          : {sum(1 for d in all_documents if d.metadata['element_type'] == 'Table')}")


Found 5 PDF files

Processing: Oracle.pdf


Could not get FontBBox from font descriptor because None cannot be parsed as 4 floats


Processing: Meta.pdf
Processing: Tesla.pdf
Processing: Nvidia.pdf
Processing: Apple.pdf

✅ Extraction complete!
Total Documents : 1043
Text Blocks     : 755
Tables          : 288


In [4]:
import yaml

def to_okf_concept(doc):
    company_guess = doc.metadata.get("file_name", "Unknown").split(".")[0].replace("_", " ")
    frontmatter = {
        "type": "FinancialTable" if doc.metadata.get("element_type") == "Table" else "FinancialText",
        "company": company_guess,
        "source_file": doc.metadata.get("file_name", "Unknown"),
        "page": doc.metadata.get("page_number", "?"),
    }
    fm_str = yaml.dump(frontmatter, sort_keys=False)
    doc.page_content = f"---\n{fm_str}---\n\n{doc.page_content.strip()}"
    return doc

all_documents = [to_okf_concept(d) for d in all_documents]
print(f"✅ Wrapped {len(all_documents)} documents as OKF concept blocks (frontmatter + content)")


✅ Wrapped 1043 documents as OKF concept blocks (frontmatter + content)


In [5]:
import subprocess
import sys

# 1. Wipe ALL related cached modules
to_remove = [k for k in sys.modules if any(x in k for x in 
    ["sentence", "langchain_huggingface", "huggingface", "langchain_core", "langchain"])]
for mod in to_remove:
    del sys.modules[mod]

# 2. Reinstall both together
subprocess.run(["pip", "install", "-q", "--no-cache-dir",
    "sentence-transformers==3.0.1",
    "langchain-huggingface==1.2.2"], check=True)

# 3. Verify sentence_transformers loads directly first
import sentence_transformers
print("sentence_transformers version:", sentence_transformers.__version__)

# 4. Now load HuggingFaceEmbeddings fresh
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en-v1.5",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"normalize_embeddings": True}
)
print("✅ Embedding model loaded!")

/usr/local/lib/python3.12/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
2026-08-30 07:03:51.348755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1788073431.595000     148 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1788073431.662195     148 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1788073432.239648     148 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the sam

sentence_transformers version: 3.0.1


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!


In [6]:
pip install langchain-experimental

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.2/211.2 kB 5.5 MB/s eta 0:00:00a 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [7]:
# CELL 6: Faster Semantic Chunking (GPU Optimized)
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings
import torch
import gc

# Clear memory
torch.cuda.empty_cache()
gc.collect()

print(f"Current GPU memory: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# Use GPU with small batch size
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cuda"},
    encode_kwargs={
        "normalize_embeddings": True,
        "batch_size": 2          # Small batch = less memory
    }
)

TARGET_COMPANIES = ["tesla","nvidia"]

def _is_target_company(doc):
    fname = doc.metadata.get("file_name","").lower()
    return any(c in fname for c in TARGET_COMPANIES)

target_docs = [doc for doc in all_documents if _is_target_company(doc)]
skipped_docs = [doc for doc in all_documents if not _is_target_company(doc)]

table_docs = [doc for doc in target_docs if doc.metadata.get("element_type") == "Table"]
text_docs = [doc for doc in target_docs if doc.metadata.get("element_type") == "Text"]

print(f"Target docs (Tesla + Nvidia): {len(target_docs)}  |  Skipped (other companies): {len(skipped_docs)}")
print(f"  -> Tables: {len(table_docs)} | Text: {len(text_docs)}")

semantic_splitter = SemanticChunker(
    embeddings = embeddings,
    breakpoint_threshold_type = "percentile",
    breakpoint_threshold_amount = 90,
)

print("🔄 Performing semantic chunking on GPU (Tesla & Nvidia only)...")
text_chunks = semantic_splitter.split_documents(text_docs)

chunks = table_docs + text_chunks

del embeddings, semantic_splitter
torch.cuda.empty_cache()
gc.collect()

print(f"✅ Done!")
print(f"Tables: {len(table_docs)} | Text Chunks: {len(text_chunks)} | Total: {len(chunks)}")
print(f"GPU memory now: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
print(f"ℹ️  Note: {len(skipped_docs)} documents from other companies (Oracle, Meta, Apple) were excluded from chunking/vectorstore.")



/tmp/ipykernel_148/3161353956.py:2: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.text_splitter import SemanticChunker


Current GPU memory: 0.41 GB


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Target docs (Tesla + Nvidia): 390  |  Skipped (other companies): 653
  -> Tables: 118 | Text: 272
🔄 Performing semantic chunking on GPU (Tesla & Nvidia only)...
✅ Done!
Tables: 118 | Text Chunks: 701 | Total: 819
GPU memory now: 0.42 GB
ℹ️  Note: 653 documents from other companies (Oracle, Meta, Apple) were excluded from chunking/vectorstore.


In [8]:
sample_embedding = embedding_model.embed_query(
    chunks[0].page_content
)

print("Embedding dimension:", len(sample_embedding))

Embedding dimension: 768


In [9]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents = chunks,
    embedding = embedding_model,
    persist_directory = "./financial_db" 
)

print("✅ Vector Store Created and Persisted")

✅ Vector Store Created and Persisted


In [10]:
query = "What is NVIDIA's total revenue?"
results = vectorstore.similarity_search(query,k = 3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(f"Source: {doc.metadata['file_name']} | Page: {doc.metadata.get('page_number')}")
    print(f"Type: {doc.metadata['element_type']}")
    print(doc.page_content[:500] + "..." if len(doc.page_content) > 500 else doc.page_content)


--- Result 1 ---
Source: Nvidia.pdf | Page: 94
Type: Text
---
type: FinancialText
company: Nvidia
source_file: Nvidia.pdf
page: 94
---

Table of Contents
NVIDIA Corporation and Subsidiaries
Notes to the Consolidated Financial Statements
(Continued)
We recognized revenue of $974 million and $729 million in fiscal years 2026 and 2025, respectively, that were included in the prior year
end deferred revenue balance. As of January 25, 2026, revenue related to remaining performance obligations from contracts greater than one year in length was $2.3
billion, ...

--- Result 2 ---
Source: Nvidia.pdf | Page: 57
Type: Table
---
type: FinancialTable
company: Nvidia
source_file: Nvidia.pdf
page: 57
---

Revenue | 100.0 | % |  | 100.0 | %
Cost of revenue | 28.9 |  |  | 25.0 | 
Gross profit | 71.1 |  |  | 75.0 | 
Operating expenses |  |  |  |  | 
Research and development | 8.6 |  |  | 9.9 | 
Sales, general and administrative | 2.1 |  |  | 2.7 | 
Total operating expenses | 10.7 |  |  | 12.6 | 
Opera

In [11]:
!pip install -q langchain langchain-community

In [12]:
import os
import re
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# ── ChatMessageHistory: moved to langchain-core in 1.x
from langchain_core.chat_history import InMemoryChatMessageHistory
from types import SimpleNamespace
from langchain_core.tools import tool

# Cross Encoder for reranking
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Per-session memory store
_memory_store = {}

def get_memory(company=None, session_id="default"):
    key = (company.lower().strip() if company else None, session_id)
    if key not in _memory_store:
        _memory_store[key] = InMemoryChatMessageHistory()
    return _memory_store[key]

def _strip_sec_header(text: str) -> str:
    if "[SEC FILING DATA]" not in text:
        return text.strip()
    parts = text.split("---\n", maxsplit=2)
    return parts[2].strip() if len(parts) >= 3 else text.strip()

def _clean_text(text):
    markers = ["<think>", "</think>", "**Final Answer**", "Final Answer:", "Changes made:"]
    for m in markers:
        if m in text:
            text = text.split(m)[0]
    return re.sub(r'\n+(I have|Note that|Please note).*', '', text,
                  flags=re.IGNORECASE | re.DOTALL).strip()

# ============================================================
# Calculator Tool
# ============================================================
@tool
def calculator(expression: str) -> str:
    """
    Use this tool for ANY mathematical calculation in financial analysis.
    Especially for percentage changes, differences, ratios, margins, and growth rates.
    
    Examples of good inputs:
    - ((215938 - 130497) / 130497) * 100
    - 193737 - 115186
    - (153463 / 215938) * 100
    """
    try:
        expr = expression.replace(",", "").replace("$", "").strip()
        expr = expr.replace("%", "/100")
        
        # Safety: only allow math characters
        if not re.match(r'^[\d\s\+\-\*\/\(\)\.]+$', expr):
            return "Error: Invalid characters in expression"
        
        result = eval(expr)
        return f"{result:.4f}" if isinstance(result, float) else str(result)
    except Exception as e:
        return f"Calculation Error: {str(e)}"

print("✅ Core utilities + Calculator tool loaded")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Core utilities + Calculator tool loaded


In [ ]:
import subprocess, sys
subprocess.run(["pip", "install", "-q", "langchain-groq"], check=True)

import os
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"] = "API_kEY"

chat_llm = ChatGroq(
    model = "qwen/qwen3.8-27b",
    temperature = 0.4,
    max_tokens = 1700,
    model_kwargs = {"top_p": 0.90},
)

llm = chat_llm

print("✅ Using qwen/qwen3.8-27b")
print("`llm`      -> used by financial_rag()")
print("`chat_llm` -> used by the tool-calling agent")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.7 MB/s eta 0:00:00
✅ Using qwen/qwen3.8-27b
`llm`      -> used by financial_rag()
`chat_llm` -> used by the tool-calling agent


In [14]:
tools = [calculator]
chat_llm_with_tools = chat_llm.bind_tools(tools)

print("✅ Calculator tool bound to chat_llm_with_tools")

✅ Calculator tool bound to chat_llm_with_tools


In [15]:
# Hybrid Retrieval (Semantic + BM25)
def hybrid_retrieval(query, vectorstore, company=None, k=50):
    # 1. Semantic search
    results = vectorstore.similarity_search_with_score(query, k=k * 3 if company else k)

    semantic_list = []
    for doc, score in results:
        if company and company.lower() not in doc.metadata.get("source", "").lower():
            continue
        semantic_list.append((doc, 1.0 / (1.0 + score)))

    # 2. BM25 keyword search
    all_data = vectorstore._collection.get(include=["documents", "metadatas"])
    filtered_texts, filtered_metas = [], []

    for text, meta in zip(all_data["documents"], all_data["metadatas"]):
        if company and company.lower() not in str(meta.get("source", "")).lower():
            continue
        filtered_texts.append(_strip_sec_header(text))
        filtered_metas.append(meta)

    bm25_list = []
    if filtered_texts:
        bm25 = BM25Okapi([t.lower().split() for t in filtered_texts])
        scores = bm25.get_scores(query.lower().split())
        top_idx = np.argsort(scores)[::-1][:k]
        for i in top_idx:
            if scores[i] > 0:
                score_norm = scores[i] / max(scores.max(), 1)
                doc = SimpleNamespace(page_content=filtered_texts[i], metadata=filtered_metas[i])
                bm25_list.append((doc, score_norm))

    # Merge semantic + BM25
    merged = {id(d[0]): d for d in semantic_list}
    for doc, score in bm25_list:
        merged[id(doc)] = (doc, merged.get(id(doc), (None, 0))[1] + score * 0.7)

    return sorted(merged.values(), key=lambda x: x[1], reverse=True)[:k]


In [16]:
#  Reranking, multimodal boost, corrective RAG
def rerank_with_cross_encoder(query, candidates, top_n=15):
    if not candidates:
        return []
    docs  = [pair[0] for pair in candidates]
    texts = [_strip_sec_header(d.page_content) for d in docs]
    scores = cross_encoder.predict([[query, t] for t in texts])
    return sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)[:top_n]


def multimodal_boost(reranked_pairs):
    boosted = []
    for doc, score in reranked_pairs:
        new_score = float(score)
        et = doc.metadata.get("element_type", "")
        text = doc.page_content.lower()

        # Strong boost for real tables
        if et == "Table":
            new_score += 0.45

        # Extra boost for income-statement / segment pages
        keywords = [
            "year ended", "consolidated statements of income",
            "revenue by", "$ in millions", "gross profit",
            "operating income","operating expense", "net income"
        ]
        
        if any(k in text for k in keywords):
            new_score += 0.25

        boosted.append((doc, new_score))
    return sorted(boosted, key=lambda x: x[1], reverse=True)


def evaluate_retrieval_quality(query, docs):
    if not docs or len(docs) < 3:
        return False
    combined_text = " ".join(
        _strip_sec_header(doc.page_content)[:1000] for doc, _ in docs[:4]
    ).lower()
    query_words = [w for w in query.lower().split() if len(w) > 3]
    if not query_words:
        return True
    overlap = sum(1 for w in query_words if w in combined_text)
    required_overlap = max(2, len(query_words) // 3)
    print(f"   Retrieval Quality: {overlap}/{required_overlap} words matched")
    return overlap >= required_overlap

print("✅ Reranking + CRAG utilities loaded")


✅ Reranking + CRAG utilities loaded


In [17]:
# Conversation memory helpers
def _build_history_text(memory, max_turns=3):
    msgs  = memory.messages
    pairs = []
    i = 0
    while i < len(msgs) - 1:
        if msgs[i].type == "human" and msgs[i + 1].type == "ai":
            pairs.append((msgs[i].content, msgs[i + 1].content))
            i += 2
        else:
            i += 1
    recent = pairs[-max_turns:]
    if not recent:
        return ""
    lines = ["Previous conversation:"]
    for turn_idx, (q, a) in enumerate(reversed(recent), 1):
        short_a = a[:500] + "…" if len(a) > 500 else a
        lines.append(f"\n[Turn {turn_idx}] User: {q}")
        lines.append(f"AI: {short_a}")
    return "\n".join(lines)

print("✅ Memory helpers loaded")


✅ Memory helpers loaded


In [18]:
# Main financial_rag() function 
def financial_rag(query: str, company: str = None, session_id: str = "default"):
    global vectorstore, embedding_model, llm

    if not all([vectorstore, embedding_model, llm]):
        return "❌ Error: vectorstore, embedding_model or llm not initialized."

    memory = get_memory(company, session_id)

    # Truncate query for clean logging (no prompt leak)
    query_preview = (query.strip()[:60] + "...") if len(query.strip()) > 60 else query.strip()
    print(f"🔍 Company: {company or 'All'} | Query: {query_preview}")

    # ── Retrieval ──────────────────────────────────────────────────────────
    candidates = hybrid_retrieval(query, vectorstore, company=company, k=40)
    reranked   = rerank_with_cross_encoder(query, candidates, top_n= 7)
    reranked   = multimodal_boost(reranked)

    # ── Corrective RAG ─────────────────────────────────────────────────────
    if not evaluate_retrieval_quality(query, reranked):
        print("⚠️  Corrective RAG triggered — widening search...")
        candidates = hybrid_retrieval(query, vectorstore, company=company, k=50)
        reranked   = rerank_with_cross_encoder(query, candidates, top_n=9)
        reranked   = multimodal_boost(reranked)

    # ── Filter & cap ───────────────────────────────────────────────────────
    filtered_docs = [
        (doc, score) for doc, score in reranked
        if not company or company.lower() in str(doc.metadata.get("source", "")).lower()
    ][:16]

    if len(filtered_docs) < 3:
        return f"❌ Not enough relevant information found for '{company}'."

    context = "\n\n---\n\n".join(_strip_sec_header(doc.page_content) for doc, _ in filtered_docs)

    # ── Pass 1: Generate Response ───────────────────────────────────────────
    print("📝 Pass 1: Generating Response")
    pass1_prompt = f"""You are a Senior Institutional Financial Analyst.

ABSOLUTE RULES — NEVER BREAK THEM:

1. Use ONLY numbers that appear VERBATIM in the Context below.
2. Use the context provided below
3. Never invent, estimate, round, or pull any number from memory or training data.
4. LABEL LOCKING (critical):
   - Every number must stay attached to the exact same label it has in the Context.
   - Operating Expenses numbers can ONLY be used for Operating Expenses.
   - Operating Income numbers can ONLY be used for Operating Income.
   - Revenue numbers can ONLY be used for Revenue.
   - Gross Profit / Gross Margin numbers can ONLY be used for Gross Profit / Gross Margin.
   - Net Income numbers can ONLY be used for Net Income.
   - Research & Development, SG&A, and other line items must also keep their own numbers.
   - Never swap or mix numbers between different metrics.

5. When you write a number, always pair it with its full correct label, for example:
   “Operating expenses were $23,076 million”
   “Operating income was $130,387 million”
   Never write a bare number and later assign it to a different metric.

6. DIRECTION RULE:
   - Before writing “increased”, “decreased”, “rose”, “declined”, “growth”, or “drop”, 
     first compare the two numbers belonging to the SAME metric.
   - Later > Earlier → must say “increased” or “rose”.
   - Later < Earlier → must say “decreased” or “declined”.
   - Never assume direction from the movement of expenses or from surrounding text.

7. If either year is missing for a metric, write exactly: “percentage change not available in the retrieved sections.”

9. If a metric is not present in the Context, write: “not disclosed in the retrieved sections.”

11. Write in clear Finincail tone

FIRST, decide the type of question:

A. If the question is mainly about financial figures, trends, revenue, margins, expenses, income, cash flow, etc.:
   → Present the key figures in a clean Markdown table with columns such as:
     | Metric                  | Earlier Year | Later Year | Change | % Change          |
     |-------------------------|--------------|------------|--------|-------------------|
   → After the table, write a structured analysis using only the numbers from the table.
   → For every % Change, show the calculation (example: ((130387-81453)/81453 × 100 = 60.1%)).

B. If the question is about architecture, technology, products (Blackwell, Rubin, etc.), strategy, risks, competition, outlook, or any non-numeric topic:
   → Do NOT create a financial table.
   → Directly write a clear, structured analysis based on the Context.
   → Only mention numbers if they are relevant and present in the Context.

**Context:**
{context}

**Question:**
{query}

**Output Format:**
Provide a structured, concise, and accurate financial analysis.

Financial Analysis:"""

    raw_pass1 = llm.invoke(pass1_prompt)
    final_response = raw_pass1.content if hasattr(raw_pass1, "content") else str(raw_pass1)
    final_response = _clean_text(final_response)


   
    # ── Memory ─────────────────────────────────────────────────────────────
    memory.add_user_message(query)
    memory.add_ai_message(final_response)

    # ── Sources ────────────────────────────────────────────────────────────
    sources = [
        f"{doc.metadata.get('file_name', 'Unknown')} | Page {doc.metadata.get('page_number', '?')}"
        for doc, _ in filtered_docs
    ]
    unique_sources = list(dict.fromkeys(sources))

    final_output = (
        f"# Financial Analysis — {company or 'All Companies'}\n\n"
        + final_response
        + "\n\n## Sources\n"
        + "\n".join(f"- {s}" for s in unique_sources)
    )

    # Debug metadata
    financial_rag._last_context = context
    financial_rag._last_sources = unique_sources
    financial_rag._debug = {
        "initial_docs": len(candidates),
        "final_docs": len(filtered_docs),
        "corrective_triggered": not evaluate_retrieval_quality(query, reranked),
    }

    return final_output

print("✅ financial_rag() is ready")


✅ financial_rag() is ready


In [19]:
# CELL 17b: BERTScore — measures how well the generated answer is grounded in retrieved context 
from bert_score import score as bert_score

_bert_log = []  # stores {query, company, precision, recall, f1} for every call

def compute_bertscore(reference_text: str, candidate_text: str):
    """
    BERTScore of candidate_text against reference_text.
    Here reference = retrieved source context, candidate = generated answer.

    Unlike BLEU, this compares contextual embeddings of tokens instead of
    exact word matches, so a well-paraphrased but accurate answer still
    scores high. Used here as a grounding/faithfulness proxy:
      - High F1  -> answer's meaning is well supported by the retrieved context
      - Low F1   -> answer may be drifting from / hallucinating beyond the context

    Returns (precision, recall, f1) as plain floats.
    """
    if not reference_text.strip() or not candidate_text.strip():
        return 0.0, 0.0, 0.0

    # BERTScore compares sentence-by-sentence internally; long inputs are fine,
    # but we truncate extremely long context purely to keep this fast.
    ref = reference_text[:4000]
    cand = candidate_text[:4000]

    P, R, F1 = bert_score(
        [cand], [ref],
        lang="en",
        model_type="distilbert-base-uncased",
        verbose=False,
    )
    return P.item(), R.item(), F1.item()


def financial_rag_with_bertscore(query: str, company: str = None, session_id: str = "default"):
    """
    Thin wrapper around financial_rag() that additionally computes and prints
    a BERTScore for the generated response (answer vs. retrieved context),
    and logs it to _bert_log for later inspection / averaging.
    """
    response = financial_rag(query, company=company, session_id=session_id)

    context = getattr(financial_rag, "_last_context", "")
    precision, recall, f1 = compute_bertscore(context, response) if context else (0.0, 0.0, 0.0)

    print(f"📊 BERTScore (answer vs. retrieved context) — Precision: {precision:.4f} | Recall: {recall:.4f} | F1: {f1:.4f}")

    _bert_log.append({
        "query": query.strip()[:80],
        "company": company or "All",
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    })

    return response


# Backward-compatible alias, in case earlier cells still call the old name
financial_rag_with_bleu = financial_rag_with_bertscore


def show_bertscore_log():
    """Pretty-print the BERTScore for every query run so far."""
    if not _bert_log:
        print("No queries logged yet.")
        return
    print(f"{'#':<3} {'Company':<10} {'Precision':<10} {'Recall':<10} {'F1':<8} Query")
    print("-" * 100)
    for i, entry in enumerate(_bert_log, 1):
        print(f"{i:<3} {entry['company']:<10} {entry['precision']:<10} {entry['recall']:<10} {entry['f1']:<8} {entry['query']}")
    avg_p = sum(e['precision'] for e in _bert_log) / len(_bert_log)
    avg_r = sum(e['recall'] for e in _bert_log) / len(_bert_log)
    avg_f1 = sum(e['f1'] for e in _bert_log) / len(_bert_log)
    print("-" * 100)
    print(f"Average -> Precision: {avg_p:.4f} | Recall: {avg_r:.4f} | F1: {avg_f1:.4f}")

print("✅ BERTScore ready — use financial_rag_with_bertscore(query, company=...) to see scores")
print("✅ Call show_bertscore_log() anytime to see all scores so far")


✅ BERTScore ready — use financial_rag_with_bertscore(query, company=...) to see scores
✅ Call show_bertscore_log() anytime to see all scores so far


In [20]:
from IPython.display import display, Markdown

query = """Provide a comprehensive financial analysis of NVIDIA using the latest SEC 10-K filing.

Focus on:
- Total revenue breakdown and year-over-year growth (Data Center vs Gaming vs Professional Visualization vs Automotive)
- Data Center segment performance, including AI infrastructure demand drivers
- Discuss What are the Gross margin trends and key factors affecting profitability
- Discuss what are the Operating expenses, operating income, and net income trends
- Cash flow generation, capital expenditures, and liquidity position
- Key financial highlights and management commentary on future outlook


Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors."""



response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Provide a comprehensive financial analysis of NVIDIA using t...
   Retrieval Quality: 20/25 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 20/25 words matched


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

📊 BERTScore (answer vs. retrieved context) — Precision: 0.7526 | Recall: 0.7875 | F1: 0.7697


# Financial Analysis — Nvidia

# NVIDIA Corporation (NVDA) Financial Analysis: Fiscal Year 2026

## 1. Executive Summary & Financial Highlights

NVIDIA’s fiscal year 2026 (ended Jan 25, 2026) demonstrates explosive top-line growth and robust profitability, driven primarily by the Data Center segment. The company transitioned from a graphics-focused entity to a dominant AI infrastructure provider.

**Key Financial Highlights (FY2026 vs. FY2025):**
*   **Total Revenue:** Increased from $130,497 million to $215,938 million.
*   **Gross Profit:** Increased from $97,858 million to $153,463 million.
*   **Operating Income:** Increased from $81,453 million to $130,387 million.
*   **Net Income:** Increased from $72,880 million to $120,067 million.
*   **Liquidity:** Cash, cash equivalents, and marketable securities stood at $62.6 billion as of January 25, 2026.

---

## 2. Revenue Breakdown and Year-Over-Year Growth

NVIDIA’s revenue growth was heavily concentrated in the Data Center segment, which accounted for the vast majority of total revenue. The following table details the revenue by end market for the last three fiscal years.

| Metric (Revenue) | FY2024 ($M) | FY2025 ($M) | FY2026 ($M) | Change ($M) | % Change |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Data Center** | $47,525 | $115,186 | $193,737 | $78,551 | 68.2% |
| **Gaming** | $10,447 | $11,350 | $16,042 | $4,692 | 41.3% |
| **Professional Visualization** | $1,553 | $1,878 | $3,191 | $1,313 | 70.0% |
| **Automotive** | $1,091 | $1,694 | $2,349 | $655 | 38.7% |
| **OEM and Other** | $306 | $389 | $619 | $230 | 59.1% |
| **Total Revenue** | **$60,922** | **$130,497** | **$215,938** | **$85,441** | **65.5%** |

*Calculations:*
*   **Data Center % Change:** ((193,737 - 115,186) / 115,186) × 100 = 68.2%
*   **Gaming % Change:** ((16,042 - 11,350) / 11,350) × 100 = 41.3%
*   **Professional Visualization % Change:** ((3,191 - 1,878) / 1,878) × 100 = 70.0%
*   **Automotive % Change:** ((2,349 - 1,694) / 1,694) × 100 = 38.7%
*   **Total Revenue % Change:** ((215,938 - 130,497) / 130,497) × 100 = 65.5%

### Data Center Segment Performance
The Data Center segment was the primary engine of growth, contributing $193,737 million in FY2026, up from $115,186 million in FY2025.
*   **Compute vs. Networking:** Within Data Center, Compute revenue was $162,361 million (up from $102,196 million) and Networking revenue was $31,376 million (up from $12,990 million).
*   **AI Infrastructure Drivers:** Management attributes the year-over-year increase to "major platform shifts – accelerated computing and AI." Specifically, revenue from Data Center computing grew 59% driven by demand for the **Blackwell computing platform**. The context notes that Blackwell architectures represented the majority of Data Center revenue.
*   **Reportable Segments:** In the "Compute & Networking" reportable segment, revenue increased by $77,286 million (67%) to $193,479 million. The "Graphics" segment (primarily Gaming) increased by $8,155 million (57%) to $22,459 million.

---

## 3. Gross Margin Trends and Profitability Factors

NVIDIA maintained exceptionally high gross margins, though there was a slight compression in FY2026 compared to FY2025.

| Metric | FY2025 | FY2026 | Change |
| :--- | :--- | :--- | :--- |
| **Gross Profit ($M)** | $97,858 | $153,463 | +$55,605 |
| **Cost of Revenue ($M)** | $32,639 | $62,475 | +$29,836 |
| **Gross Margin (%)** | 75.0% | 71.1% | -3.9 pts |

*Calculations:*
*   **Gross Profit Growth:** ((153,463 - 97,858) / 97,858) × 100 = 56.8%
*   **Cost of Revenue Growth:** ((62,475 - 32,639) / 32,639) × 100 = 91.4%

**Analysis:**
*   **Margin Compression:** The Gross Margin decreased from 75.0% in FY2025 to 71.1% in FY2026. This decline is attributable to the fact that Cost of Revenue grew at a faster rate (91.4%) than Revenue (65.5%).
*   **Key Factors:** While the specific drivers for the cost increase are not itemized in the provided text, the significant scale-up in production for the Blackwell platform and Data Center infrastructure likely contributed to higher absolute costs. Despite the percentage drop, the absolute Gross Profit increased by $55,605 million, reflecting the sheer volume of high-value AI hardware sales.

---

## 4. Operating Expenses, Operating Income, and Net Income Trends

NVIDIA demonstrated strong operating leverage, where operating income grew faster than revenue, leading to a slight expansion in operating margins.

| Metric | FY2025 ($M) | FY2026 ($M) | Change ($M) | % Change |
| :--- | :--- | :--- | :--- | :--- |
| **Total Operating Expenses** | $16,405 | $23,076 | $6,671 | 40.7% |
| **Research & Development** | $12,914 | $

## Sources
- Nvidia.pdf | Page 71
- Nvidia.pdf | Page 106
- Nvidia.pdf | Page 72
- Nvidia.pdf | Page 57
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 60
- Nvidia.pdf | Page 53
- Nvidia.pdf | Page 22

In [21]:
query = "Discuss about the architecture of Blackwell and Rubin and how they are benefits to Nvidia in depth "

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Discuss about the architecture of Blackwell and Rubin and ho...
   Retrieval Quality: 1/3 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 1/3 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7426 | Recall: 0.8199 | F1: 0.7793


# Financial Analysis — Nvidia

**Financial Analysis:**

Based on the retrieved sections, the following is a structured analysis of the NVIDIA Blackwell and Rubin architectures, their technical specifications, and their strategic benefits to the company.

### 1. NVIDIA Blackwell Architecture

**Launch and Timeline**
*   **Initial Launch:** The NVIDIA Blackwell architecture was launched in **fiscal year 2025** (also referenced as **2024** in the innovation history section) as a full set of data center scale infrastructure.
*   **Blackwell Ultra:** In **fiscal year 2026**, NVIDIA launched and scaled the **NVIDIA Blackwell Ultra** platform.
*   **Production Start:** Production units of the new Blackwell Ultra platforms, including **GB300**, began shipping in the **second quarter of fiscal year 2026**.

**Technical Composition and Design**
*   **Components:** The architecture includes **GPUs**, **CPUs**, **DPUs**, interconnects, switch chips and systems, and networking adapters.
*   **System Integration:** The design connects **36 Grace CPUs** and **72 Blackwell GPUs** in a data center scale, liquid-cooled design.
*   **Software Integration:** It leverages **Dynamo inference software** and is part of a full-stack computing platform that includes the **CUDA** development platform, **CUDA-X** libraries, and **NVIDIA AI Enterprise**.

**Performance and Efficiency Benefits**
*   **Workload Optimization:** Blackwell excels at processing cutting-edge generative AI and accelerated computing workloads with market-leading performance and efficiency.
*   **Throughput and Cost:** The Blackwell Ultra platform delivers a significant increase in token throughput and a reduction in cost per token compared to the **Hopper** generation.
*   **Capability:** It is optimized for real-time trillion-parameter inference and training, as well as agentic, reasoning, and physical AI.

**Strategic Benefit to NVIDIA**
*   **Revenue Driver:** Revenue growth in **fiscal year 2026** was driven by data center compute and networking platforms, with Blackwell architectures representing the **majority** of Data Center revenue.
*   **Market Position:** It reinforces NVIDIA’s position as a data center scale AI infrastructure company, allowing the company to deliver order-of-magnitude performance advantages relative to legacy approaches.

### 2. NVIDIA Rubin Platform

**Launch and Timeline**
*   **Unveiling:** The NVIDIA Rubin platform was unveiled in **fiscal year 2026**.
*   **Production Start:** It is expected to commence production shipments in the **second half of fiscal year 2027**.

**Technical Composition and Design**
*   **Focus:** Built specifically for **agentic AI** and **reasoning**.
*   **Capabilities:** It excels at processing multi-step problem-solving and massive long-context workflows.

**Performance and Efficiency Benefits**
*   **Cost Efficiency:** The platform delivers up to a **10x reduction in cost per token** compared to Blackwell.

**Strategic Benefit to NVIDIA**
*   **Product Cadence:** Rubin supports NVIDIA’s strategy of executing Data Center compute product introductions on a **one-year product cadence**.
*   **Future Growth:** By targeting agentic AI and reasoning, Rubin positions NVIDIA to capture emerging high-compute workloads, sustaining the demand for its accelerated computing platform.

### 3. Comparative Analysis and Strategic Implications

| Metric / Feature | Blackwell / Blackwell Ultra | Rubin |
| :--- | :--- | :--- |
| **Launch/Unveil Year** | Fiscal Year 2025 (Launch) / Fiscal Year 2026 (Ultra) | Fiscal Year 2026 (Unveil) |
| **Production Start** | Q2 Fiscal Year 2026 (Blackwell Ultra/GB300) | Second Half of Fiscal Year 2027 |
| **Primary Focus** | Generative AI, Accelerated Computing, Trillion-parameter inference | Agentic AI, Reasoning, Multi-step problem-solving |
| **Cost Efficiency** | Significant reduction in cost per token vs. Hopper | Up to 10x reduction in cost per token vs. Blackwell |
| **System Configuration** | 36 Grace CPUs + 72 Blackwell GPUs (Liquid-cooled) | Not disclosed in retrieved sections |

**Strategic Synthesis:**
NVIDIA utilizes a unified underlying architecture leveraging GPUs, CPUs, CUDA, and networking technologies to address diverse end markets including Data Center, Gaming, Professional Visualization, and Automotive. The transition from Blackwell to Rubin illustrates a continuous innovation cycle designed to outpace Moore’s Law.

*   **Revenue Impact:** The immediate financial benefit is evident in **fiscal year 2026**, where Blackwell architectures accounted for the majority of Data Center revenue.
*   **R&D Leverage:** NVIDIA has invested over **$76.7 billion** in research and development since inception. This investment allows for leveraged R&D, where shared underlying technology supports multiple multi-billion-dollar end markets.
*   **Risk Consideration:** The complexity of product transitions and sophisticated system configurations may cause delays in production and create challenges in managing supply and demand. Additionally, the availability of data centers, energy, and capital for customer buildout is crucial for sustaining future revenue growth.

## Sources
- Nvidia.pdf | Page 7
- Nvidia.pdf | Page 4
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 9

In [22]:

# CELL 26: NVIDIA — Export Control Risks

from IPython.display import display, Markdown

query = """Provide a detailed analysis of NVIDIA's export control risks, geopolitical exposure, and China-related challenges.

Focus on:
- Impact of U.S. export restrictions on products
- Licensing requirements, revenue impact, and inventory charges
- Competitive effects on China data center market discuss this in detail
- Mitigation strategies and long-term implications
- Broader supply chain and regulatory risks discuss this in detail

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 


Quote relevant sections from the Risk Factors and Business sections."""

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Provide a detailed analysis of NVIDIA's export control risks...
   Retrieval Quality: 20/23 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 20/23 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7692 | Recall: 0.7936 | F1: 0.7812


# Financial Analysis — Nvidia

**Financial Analysis: NVIDIA Export Control Risks, Geopolitical Exposure, and China-Related Challenges**

Based on the provided context from NVIDIA’s 10-K filing (pages 15, 37, 39, 41, and 43), the following is a detailed analysis of the company’s exposure to export controls, geopolitical risks, and specific challenges in the China market.

### 1. Impact of U.S. Export Restrictions on Products

NVIDIA faces significant operational constraints due to unilateral U.S. government (USG) export controls targeting GPUs and semiconductors associated with AI. These restrictions are not limited to China but have expanded to a worldwide scope, creating a complex regulatory environment.

*   **Product-Specific Restrictions:** The USG has imposed controls that restrict the export of specific high-performance data center products. The context explicitly mentions that the "AI Diffusion IFR" (Interim Final Rule) published in January 2025 would have imposed a worldwide licensing requirement on data center products, specifically naming the **H200, GB200, and GB300**. Although the USG announced in May 2025 that it would rescind the AI Diffusion IFR to implement a replacement rule, the scope and timing of this new rule remain uncertain.
*   **Technical Parameters:** The export controls are based on complex technical parameters, including:
    *   Total processing performance of a chip.
    *   "Performance density" of a chip.
    *   Interconnect bandwidth of a chip.
    *   Memory bandwidth of a chip.
*   **Network Interconnects:** Beyond GPUs, the USG may impose export controls on NVIDIA’s networking products, such as high-speed network interconnects, to limit the ability of downstream parties to create large clusters for frontier model training.

**Quote:**
> "In January 2025, the USG published the AI Diffusion IFR in the Federal Register. The IFR would have imposed a worldwide licensing requirement on our data center products, such as our H200, GB200 and GB300... In May 2025, the USG announced that it would rescind the AI Diffusion IFR and implement a replacement rule. The scope, timing, and requirements of the forthcoming rule remain uncertain." (Page 41)

### 2. Licensing Requirements, Revenue Impact, and Inventory Charges

The licensing regime introduces significant friction into NVIDIA’s sales cycle, leading to direct financial impacts including revenue loss, excess inventory, and supply charges.

*   **Licensing Burden:** The licensing process is described as cumbersome and uncertain. Licenses may be temporary and impose burdensome conditions regarding installation, maintenance, and use. This uncertainty discourages customers in China, the Middle East, and other regions from purchasing NVIDIA products, as they seek to avoid compliance risks.
*   **Revenue Impact:** The possibility of additional export controls has "negatively impacted and may in the future negatively impact demand for our products." This reduction in demand directly threatens revenue, particularly in markets outside the U.S. where NVIDIA’s supply chain is concentrated in Asia.
*   **Inventory and Supply Charges:** Reduced demand due to export controls has led to, and could lead to, **excess inventory** or cause NVIDIA to incur **related supply charges**. A specific example cited is the **H20** product, where NVIDIA recently experienced excess inventory and purchase obligations because the product became subject to new unilateral export controls before it was ready for market.

**Quote:**
> "Reduced demand due to export controls has and could in the future lead to excess inventory or cause us to incur related supply charges... resulting in excess inventory and purchase obligations as we recently experienced with the H20." (Pages 39, 43)

### 3. Competitive Effects on China Data Center Market

The most severe competitive impact is observed in the China data center market, where NVIDIA is effectively foreclosed from competing.

*   **Effective Foreclosure:** As of the end of fiscal year 2026, NVIDIA states it was "effectively foreclosed from competing in China's data center computing/compute market." This is because the company is unable to create and deliver a competitive product that receives approval from both the USG and the Chinese government.
*   **Benefit to Competitors:** This foreclosure has allowed competitors to build larger developer and customer ecosystems, challenging NVIDIA worldwide. The context notes that export controls have "disproportionate impact on NVIDIA" and may disadvantage it against competitors selling chips outside the scope of such controls.
*   **Customer "Design-Out":** Export controls have encouraged customers outside China and other impacted regions to "design-out" certain U.S. semiconductors from their products to reduce compliance burden and risk. This structural shift in customer behavior poses a long-term threat to NVIDIA’s market share.
*   **China Government Stance:** The Chinese government has actively encouraged customers to purchase from China-based competitors and discouraged the purchase, import, or use of NVIDIA’s data center products, including any China-specific product designed to comply with U.S. export controls.

**Quote:**
> "As of the end of fiscal year 2026, we were effectively foreclosed from competing in China's data center computing/compute market, and our effective foreclosure from the China market helped our competitors build larger developer and customer ecosystems to challenge us worldwide." (Page 41)

### 4. Mitigation Strategies and Long-Term Implications

NVIDIA’s mitigation strategies are currently constrained by the conflicting requirements of U.S. and Chinese regulations.

*   **Supply Chain Resiliency:** NVIDIA is working to enhance the resiliency and redundancy of its supply chain, which is currently concentrated in Asia. However, new export controls could limit alternative manufacturing locations, negating these efforts.
*   **Degraded Products:** To comply with U.S. export controls, NVIDIA was required to offer "degraded products" to the Chinese market. However, this strategy has backfired legally in China. On September 15, 2025, China’s antitrust regulators published a preliminary finding that this practice "discriminated unfairly against customers in the China market" and violated the terms of China’s approval of NVIDIA’s Mellanox acquisition.
*   **Long-Term Implications:**
    *   **Regulatory Penalties:** If regulators conclude NVIDIA failed to fulfill the terms of the Mellanox acquisition or violated Chinese law, the company could face financial penalties, restrictions on its ability to conduct business, or orders regarding its networking business in China.
    *   **Lost Opportunity:** The inability to return to the China market with an approved product will result in a "material and adverse impact on our business, operating results, and financial condition."
    *   **R&D Constraints:** Additional export restrictions include "deemed export control limitations" that negatively impact the ability of NVIDIA’s research and development teams to execute its roadmap in a timely manner.

**Quote:**
> "On September 15, 2025, China’s antitrust regulators published their preliminary finding that our compliance with applicable U.S. export controls, which required us to offer degraded products to the Chinese market, discriminated unfairly against customers in the China market and therefore violated the terms of China’s approval of our Mellanox acquisition." (Page 39)

### 5. Broader Supply Chain and Regulatory Risks

The geopolitical landscape extends beyond China, affecting NVIDIA’s global operations and regulatory standing.

*   **Supply Chain Disruption:** Export controls may disrupt NVIDIA’s supply and distribution chain for a substantial portion of its products, which are warehoused in and distributed from **Hong Kong**. This disruption impacts the ability to serve demand in markets outside China, including for non-data center products.
*   **Global Regulatory Scrutiny:** NVIDIA’s position in AI has led to increased interest from regulators worldwide, including the European Union, the United States, the United Kingdom, South Korea, Japan, and China.
    *   **Antitrust Inquiries:** The

## Sources
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 39
- Nvidia.pdf | Page 15
- Nvidia.pdf | Page 41
- Nvidia.pdf | Page 37

In [23]:
query = """Evaluate NVIDIA's  positioning across its markets.

Focus on:
- Competition in Data Center (AMD, Intel, custom ASICs from hyperscalers)
- Discuss about Gaming GPU competition 
- Professional Visualization and Automotive segments
- Overall technology leadership in GPUs, CUDA, networking, and software
- Barriers to entry and ecosystem strength

Discuss strengths, weaknesses, and investor implications with references from the filing."""

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Evaluate NVIDIA's  positioning across its markets.

Focus on...
   Retrieval Quality: 18/14 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 18/14 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7482 | Recall: 0.7803 | F1: 0.7639


# Financial Analysis — Nvidia

**Financial Analysis: NVIDIA Market Positioning and Competitive Landscape**

Based on the retrieved sections of NVIDIA’s Annual Report on Form 10-K, the following analysis evaluates the company’s positioning across its key market segments, competitive dynamics, and strategic barriers to entry.

### 1. Competitive Landscape by Segment

#### Data Center and AI Infrastructure
NVIDIA positions itself as a "data center scale AI infrastructure company" providing a complete, end-to-end accelerated computing platform for AI, addressing both training and inferencing.
*   **Competitors:** The filing identifies significant competition from:
    *   **Hardware/Software Suppliers:** Advanced Micro Devices, Inc. (AMD), Huawei Technologies Co. Ltd. (Huawei), and Intel Corporation (Intel).
    *   **Hyperscalers/Cloud Providers:** Alibaba Group, Alphabet Inc., Amazon Inc. (Amazon), Baidu Inc., Huawei, and Microsoft Corporation. These entities design internal hardware and software incorporating accelerated or AI computing functionality.
    *   **Network Competitors:** AMD, Arista Networks, Broadcom, Cisco Systems, Inc., Hewlett Packard Enterprise Company, Huawei, Intel, Lumentum Holdings Inc., and Marvell Technology, Inc.
*   **Market Position:** NVIDIA’s Blackwell architectures represented the majority of Data Center revenue in fiscal year 2026. The company claims its full-stack innovation approach delivers "order-of-magnitude performance advantages relative to legacy approaches."
*   **Risks:** Competition is expected to increase from existing and new entrants offering lower-priced products or better performance. Additionally, export controls may disadvantage NVIDIA against competitors selling chips outside the scope of U.S. controls, potentially encouraging customers to "design-out" U.S. semiconductors.

#### Gaming
NVIDIA serves the gaming market with GeForce RTX GPUs, GeForce NOW cloud gaming, and SoCs for consoles.
*   **Competitors:** The filing lists suppliers of hardware and software for discrete and integrated GPUs, including AMD, Huawei, and Intel. It also notes competition from companies with internal teams designing SoC products, such as Tesla, Inc.
*   **Technology Leadership:** The company introduced the NVIDIA Blackwell GeForce RTX 50 Series family in fiscal year 2025. Key differentiators include:
    *   **Neural Graphics:** Combines AI models with traditional rendering to boost performance and image quality.
    *   **DLSS (Deep Learning Super Sampling):** A new transformer model architecture that boosts frame rates while generating high-quality images.
    *   **Tensor Cores:** Enable on-device AI applications.
*   **Market Drivers:** Growth is propelled by high-production-value games, eSports, social connectivity, and the rise of streamers and creators. The market is expanding due to the growing population of live streamers, broadcasters, artists, and creators.

#### Professional Visualization
NVIDIA targets this segment with RTX PRO GPUs, working closely with Independent Software Vendors (ISVs) to optimize offerings.
*   **Competitors:** The filing does not explicitly list specific competitors for this segment in the provided text, but notes that leading 3D design and content creation applications developed by ecosystem partners support RTX.
*   **Technology Leadership:** These GPUs leverage the same Tensor Core technology found in Data Center solutions, enhancing AI and data processing capabilities for workflows in design, engineering, and digital content creation.
*   **Market Drivers:** The increasing number of generative and agentic AI applications is driving demand for enhanced AI capabilities in workstation-class GPUs, particularly for on-premises enterprise deployments.

#### Automotive
*   **Competitors:** The filing lists suppliers of hardware and software for SoC products used in servers or embedded into automobiles, including Ambarella, Inc., AMD, Broadcom, Intel, Qualcomm Incorporated, Renesas Electronics Corporation, and Samsung. It also mentions companies with internal teams designing SoC products, such as Tesla, Inc.
*   **Positioning:** NVIDIA addresses this market with a unified underlying architecture leveraging GPUs, CPUs, CUDA, and networking technologies.

### 2. Technology Leadership and Ecosystem Strength

*   **Full-Stack Innovation:** NVIDIA leverages innovation across architecture, chip design, system, interconnect, algorithm, and software layers. This approach allows the company to support multiple multi-billion-dollar end markets with shared underlying technology.
*   **Unified Architecture:** The company uses a unified underlying architecture with GPUs, CPUs, CUDA, and networking technologies as fundamental building blocks. This programmable nature enables leveraged investments in R&D.
*   **Software Ecosystem:**
    *   **NVIDIA AI Enterprise:** A comprehensive software suite for production-grade generative AI, including:
        *   **NVIDIA NIM:** Increases token throughput using industry-leading open and proprietary models.
        *   **NVIDIA NeMo:** A complete solution for curating, fine-tuning, reinforcement learning, evaluating, and safeguarding domain-adapted models.
        *   **AI Blueprints:** Pre-built, runnable templates for building, optimizing, and deploying AI agents.
    *   **Application Support:** NVIDIA computing supports 6,000 applications, ranging from climate prediction to genomics.
*   **Market Penetration:** Including GPUs and networking, NVIDIA powers over 78% of the supercomputers on the global TOP500 list, including 9 of the top 10 systems on the Green500 list.

### 3. Barriers to Entry and Risks

*   **Intellectual Property:** NVIDIA relies on patents, trademarks, trade secrets, nondisclosure agreements, and licensing arrangements to protect its IP.
*   **Regulatory and Export Controls:**
    *   **Export Controls:** May disrupt supply and distribution chains, particularly for products warehoused in Hong Kong. Controls on data center GPUs may negatively impact demand for networking products.
    *   **Compliance Costs:** Increased compliance costs are expected due to changes in antitrust legislation and regulation.
    *   **Regulatory Scrutiny:** NVIDIA’s position in AI has led to increased interest from regulators worldwide, including the EU, US, UK, South Korea, Japan, and China. The French Competition Authority is conducting an ongoing inquiry into competition in the graphics card and CSP market.
    *   **Customer Behavior:** Export controls may encourage customers to "design-out" U.S. semiconductors to reduce compliance burden and risk, potentially harming NVIDIA’s market position.

### 4. Investor Implications

*   **Strengths:**
    *   Dominant position in AI infrastructure, with Blackwell architectures driving the majority of Data Center revenue in FY2026.
    *   Strong ecosystem with 6,000 supported applications and over 78% share of TOP500 supercomputers.
    *   Full-stack innovation providing order-of-magnitude performance advantages.
    *   Diversified revenue streams across Data Center, Gaming, Professional Visualization, and Automotive.
*   **Weaknesses/Risks:**
    *   Intense competition from AMD, Intel, Huawei, and hyperscalers (Amazon, Microsoft, etc.) who are developing internal AI hardware and software.
    *   Regulatory and export control risks, particularly regarding China and other regions, which could limit market access and increase compliance costs.
    *   Dependence on the availability of data centers, energy, and capital for customer buildout, which are subject to multi-year regulatory, technical, and construction challenges.
    *   Potential for customers to design out U.S. semiconductors due to export control compliance burdens.

### Conclusion

NVIDIA maintains a strong market position driven by its full-stack AI platform, technological leadership in GPUs and networking, and a robust software ecosystem. However, the company faces significant headwinds from intensifying competition, particularly from hyperscalers and traditional semiconductor rivals, as well as regulatory and export control risks that could constrain its global market access and financial performance. Investors should monitor the impact of export controls on Data Center revenue and the pace of competition in AI infrastructure.

## Sources
- Nvidia.pdf | Page 9
- Nvidia.pdf | Page 12
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 7
- Nvidia.pdf | Page 5

In [24]:
query = """Analyze NVIDIA's long-term corporate strategy, key risks, and growth outlook.

Focus on:
- Platform strategy (hardware + software + ecosystem)
- Expansion into AI, robotics, autonomous driving, and professional visualization
- Supply chain, manufacturing, and capacity risks
- Human capital, R&D investment, and innovation approach
- Major risks from the Risk Factors section and mitigation efforts

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 


Provide a balanced view with exact quotes and key takeaways for long-term investors."""


response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Analyze NVIDIA's long-term corporate strategy, key risks, an...
   Retrieval Quality: 22/22 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 22/22 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7674 | Recall: 0.7622 | F1: 0.7648


# Financial Analysis — Nvidia

**Financial Analysis: NVIDIA Long-Term Corporate Strategy, Risks, and Growth Outlook**

**Classification:** Non-numeric/Strategic Analysis (Type B)
*Note: The provided Context contains qualitative strategic descriptions, risk factors, and business model details. It does not contain specific financial figures (Revenue, Operating Expenses, Net Income, etc.) for fiscal years. Therefore, no financial table is generated, and no percentage changes are calculated. All analysis below is derived strictly from the textual evidence provided.*

### 1. Platform Strategy: Hardware, Software, and Ecosystem Integration

NVIDIA’s core competitive advantage lies in its "full-stack innovation approach," which integrates hardware, software, and ecosystem components to deliver "order-of-magnitude performance advantages relative to legacy approaches."

*   **Unified Architecture:** The company leverages a unified underlying architecture across diverse end markets (Data Center, Gaming, Professional Visualization, and Automotive). This architecture is built on "GPUs, CPUs, CUDA and networking technologies as the fundamental building blocks."
*   **Full-Stack AI Solutions:** NVIDIA provides a "complete, end-to-end accelerated computing platform for AI, addressing both training and inferencing." This includes:
    *   **Hardware:** "Full-stack data center-scale compute and networking solutions across processing units, interconnects, systems, and software." The company explicitly states it offers "all three major processing units in AI servers – GPUs, CPUs, and DPUs."
    *   **Software:** The "NVIDIA AI Enterprise" suite is designed to simplify the development and deployment of production-grade generative AI. Key components include:
        *   **NVIDIA NIM:** Increases token throughput using industry-leading open and proprietary models.
        *   **NVIDIA NeMo:** A solution for curating, fine-tuning, reinforcement learning, evaluating, and safeguarding domain-adapted models.
        *   **AI Blueprints:** Pre-built, runnable templates for building, optimizing, and deploying AI agents while preserving privacy.
*   **Ecosystem Leverage:** The strategy relies on a "large and expanding ecosystem." The programmable nature of the architecture allows NVIDIA to make "leveraged investments in research and development," supporting "several multi-billion-dollar end markets with shared underlying technology" through software stacks developed internally or by third-party partners.

**Key Takeaway for Investors:** NVIDIA is not merely a chipmaker but a platform company. The integration of hardware (GPUs/CPUs/DPUs) with proprietary software (CUDA, AI Enterprise) creates high switching costs and deepens customer lock-in, particularly in the AI sector.

### 2. Expansion into AI, Robotics, Autonomous Driving, and Professional Visualization

NVIDIA has expanded beyond its original focus on PC graphics into "several other large and important computationally intensive fields."

*   **AI and Data Center:** The company describes itself as "a data center scale AI infrastructure company reshaping all industries." Revenue growth in fiscal year 2026 was driven by "data center compute and networking platforms for accelerated computing and AI solutions." The "Blackwell architectures represented the majority of our Data Center revenue," indicating a successful transition to next-generation hardware.
*   **Autonomous Driving and Robotics:** NVIDIA offers the "DRIVE" platform, which includes:
    *   "DRIVE OS" (in-vehicle operating system).
    *   A "reference sensor set that supports full self-driving capability."
    *   An "open, modular DRIVE software platform for autonomous driving, mapping, and parking services."
    *   "Intelligent in-vehicle experiences."
*   **Professional Visualization and Other Fields:** The company leverages its GPU architecture for "scientific computing, AI, data science, autonomous vehicles, robotics, and digital twin applications."

**Key Takeaway for Investors:** The diversification into autonomous driving (DRIVE) and professional visualization reduces reliance on any single end-market. The dominance of Blackwell in Data Center revenue suggests that the AI infrastructure cycle is currently the primary growth engine.

### 3. Supply Chain, Manufacturing, and Capacity Risks

The Context highlights significant structural risks related to the physical infrastructure required to support NVIDIA’s growth.

*   **Energy and Data Center Constraints:** "The availability of data centers, energy, and capital to support the buildout of NVIDIA AI infrastructure by our customers and partners is crucial." Any shortage of these resources "could impact our future revenue and financial performance."
*   **Energy Expansion Challenges:** "Expanding energy capacity to meet demand is a complex, multi-year process that involves significant regulatory, technical, and construction challenges."
*   **Capital Access:** "Access to capital can be particularly constrained for less-capitalized companies, which may face difficulties securing financing for large-scale infrastructure projects." This implies that while NVIDIA itself may have capital access, its customers (hyperscalers, startups) may face bottlenecks that limit demand realization.

**Key Takeaway for Investors:** Growth is not solely dependent on NVIDIA’s ability to produce chips, but on the broader ecosystem’s ability to build data centers and secure energy. Regulatory and construction hurdles in energy expansion represent a long-term tail risk to revenue growth.

### 4. Human Capital, R&D Investment, and Innovation Approach

*   **R&D Strategy:** NVIDIA’s success depends on its ability to "develop or secure access to new products and technologies through investments in research and development." The company notes it has "invested in research and development in markets where we have a limited operating history, which may not produce meaningful revenue for several years, if at all."
*   **Innovation Approach:** The company aims to "deliver continued performance leaps that outpace Moore’s Law by leveraging innovation across the architecture, chip design, system, interconnect, algorithm, and software layers."
*   **Adaptability:** Success requires the ability to "timely identify industry changes, adapt our strategies, and develop new or enhance and maintain existing products and technologies that meet the evolving needs of our markets." This includes addressing "unexpected shifts in industry standards or disruptive technological innovations that could render our products incompatible with those developed by other companies."

**Key Takeaway for Investors:** NVIDIA’s R&D is high-risk, high-reward. Investments in new markets (like autonomous driving) may not yield immediate revenue. The ability to outpace Moore’s Law through full-stack innovation is critical to maintaining its competitive moat.

### 5. Major Risks from Risk Factors and Mitigation Efforts

*   **Regulatory and Antitrust Risks:**
    *   **Risk:** "Our position in markets relating to AI has led to increased interest in our business from regulators worldwide, including the European Union, the United States, the United Kingdom, South Korea, Japan, and China."
    *   **Specific Actions:** The "French Competition Authority collected information from us regarding our business and competition in the graphics card and CSP market." NVIDIA has received "broad requests for information from competition regulators... regarding our sales of GPUs and other NVIDIA products, our efforts to allocate supply, foundation models and our investments, partnerships and other agreements with companies developing foundation models."
    *   **Mitigation/Outlook:** The company expects to "receive additional requests for information in the future." There is no explicit mitigation strategy detailed in the text other than compliance with "evolving and prevailing customer and industry safety, security, reliability expectations, and compliance standards."
*   **Technological Obsolescence:**
    *   **Risk:** "Our accelerated computing platforms experience rapid changes in technology, customer requirements, competitive products, and industry standards." Failure to adapt could "render our products incompatible with those developed by other companies."
    *   **Mitigation:** The company relies on its "full-stack innovation approach" and "leveraged investments in research and development" to maintain "order-of-magnitude performance advantages."
*   **Customer Concentration and Supply Allocation:**
    *   **Risk:** Regulators are specifically inquiring about "our efforts to allocate supply." This suggests that supply constraints may be a point of regulatory scrutiny.
    *   **Mitigation:** The company aims to "manage product and software lifecycles to maintain customer and end-user satisfaction"

## Sources
- Nvidia.pdf | Page 18
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 9

In [25]:
query = "explain the supply chain risk of Nvidia in more depth like who are the key suppliers and what they supply and why it is cruical for Nvidia and also tell based on context suggest improvements in depth."

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: explain the supply chain risk of Nvidia in more depth like w...
   Retrieval Quality: 10/7 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 10/7 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7662 | Recall: 0.7857 | F1: 0.7758


# Financial Analysis — Nvidia

**Question Type:** B (Non-numeric topic: Supply chain risks, strategy, and operations).

**Financial Analysis:**

Based on the provided Context, the specific names of key suppliers and the exact items they supply are **not disclosed in the retrieved sections**. Therefore, a detailed breakdown of specific vendor identities and their individual supply contributions cannot be provided. However, the Context provides significant detail regarding the structural, regulatory, and operational risks associated with Nvidia’s supply chain.

### 1. Supply Chain Risk Profile

The Context highlights that Nvidia’s supply chain is heavily exposed to geopolitical and regulatory risks, particularly concerning export controls. The key risks identified are:

*   **Geographic Concentration and Disruption:** A substantial portion of Nvidia’s products are warehoused in and distributed from **Hong Kong**. Export controls threaten to disrupt this specific supply and distribution chain, which is critical for global logistics.
*   **Regulatory Volatility (Export Controls):**
    *   **Scope:** Controls target GPUs, semiconductors associated with AI, and networking products (such as high-speed network interconnects).
    *   **Impact on Operations:** These controls restrict the use, resale, repair, or transfer of products. This has historically and may in the future negatively impact demand in China, Europe, Latin America, and Southeast Asia.
    *   **Compliance Burden:** Repeated changes in export control rules impose significant compliance burdens on both Nvidia and its customers.
    *   **Design-Out Risk:** Customers outside China and other impacted regions are encouraged to "design-out" U.S. semiconductors (including Nvidia’s) to reduce compliance risk and ensure global market access.
    *   **Competitive Disadvantage:** Export controls may disadvantage Nvidia against competitors whose chips are outside the scope of such controls, potentially encouraging overseas governments to request purchases from competitors.
*   **Inventory and Purchase Obligations:**
    *   The Context notes that by the time a new product is ready for market, it may be subject to new unilateral export controls.
    *   This timing mismatch results in **excess inventory** and **purchase obligations**, a specific issue recently experienced with the **H20** product.
    *   Reduced demand due to export controls can lead to further excess inventory or related supply charges.
*   **Supplier and Customer Insolvency:**
    *   There is a risk of insolvency among key suppliers, distributors, customers, Cloud Service Providers (CSPs), and data center providers.
    *   Adverse developments in financial institutions (e.g., bank failures) could impact customers' ability to fulfill payment obligations and vendors' ability to fulfill contractual obligations.
*   **Integration and Operational Risks:**
    *   **Acquisition Integration:** Integrating acquired systems (e.g., Mellanox) involves challenges such as lengthy and costly systems integration, delays in purchasing and shipping products, and difficulties with electronic data interchange with key suppliers and customers.
    *   **Supply Commitments:** There is a risk that suppliers may be unable to deliver on their supply commitments to Nvidia, or that customers/licensees may be unable to supply products to end users.

### 2. Criticality to Nvidia

The supply chain is critical to Nvidia for the following reasons:

*   **Revenue Dependency:** The Context states that adverse supply chain developments could "substantially reduce our revenue." The inability to serve demand in key markets (including non-data center products and markets outside China) directly impacts financial results.
*   **Product Roadmap Execution:** Additional export restrictions, including deemed export control limitations, negatively impact the ability of Nvidia’s research and development teams to execute their roadmap in a timely manner.
*   **Cost Structure:** Disruptions and compliance burdens increase costs. The Context mentions "increased supply, employee, facilities and infrastructure costs" as factors that have reduced and may reduce margins.
*   **Market Position:** The risk of customers designing out Nvidia’s products or governments mandating the use of competitors’ products threatens Nvidia’s market position and long-term competitiveness.

### 3. Suggested Improvements (Based on Context)

While the Context does not explicitly list "improvements," it identifies specific pain points and risks that imply areas for strategic focus or mitigation:

*   **Diversification of Distribution Hubs:** Given the heavy reliance on Hong Kong for warehousing and distribution, diversifying logistics hubs to reduce geographic concentration risk would mitigate the impact of regional export controls or disruptions.
*   **Enhanced Regulatory Agility:** The Context highlights the burden of "repeated changes in export control rules." Improving internal compliance infrastructure to rapidly adapt to new unilateral or multilateral controls could reduce the lag time that leads to excess inventory (as seen with the H20).
*   **Supply Chain Resilience and Redundancy:** To address the risk of suppliers failing to deliver or becoming insolvent, establishing more robust supplier qualification processes and potentially multi-sourcing critical components could mitigate single-point-of-failure risks.
*   **Customer Education and Compliance Support:** Since customers are "designing out" U.S. semiconductors to reduce their own compliance burden, Nvidia could improve its support for customers navigating export controls, potentially through clearer guidance or compliance tools, to retain market share in impacted regions.
*   **Inventory Management Optimization:** Given the risk of excess inventory due to regulatory timing mismatches, implementing more dynamic inventory management strategies that account for regulatory volatility could reduce write-downs and supply charges.
*   **Strengthening Financial Counterparty Risk Management:** Given the risks of customer and supplier insolvency, enhancing credit risk monitoring and diversifying the customer base could reduce exposure to financial distress in key partners.

**Note:** The Context does not provide specific financial figures (e.g., revenue, expenses, inventory levels) related to these supply chain risks. Therefore, no financial table or percentage change calculations are applicable.

## Sources
- Nvidia.pdf | Page 27
- Nvidia.pdf | Page 43
- Nvidia.pdf | Page 34
- Nvidia.pdf | Page 39
- Nvidia.pdf | Page 18
- Nvidia.pdf | Page 24

In [26]:
query = "Discuss about operating expenses(R&D and SG&A) of Nvidia in depth and also discuss about trends"

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Discuss about operating expenses(R&D and SG&A) of Nvidia in ...
   Retrieval Quality: 2/3 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 2/3 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7775 | Recall: 0.8168 | F1: 0.7967


# Financial Analysis — Nvidia

### Financial Analysis: NVIDIA Operating Expenses (R&D and SG&A)

Based on the retrieved financial statements for the fiscal years ended January 25, 2026, and January 26, 2025, the following table details the composition and trends of NVIDIA’s operating expenses.

| Metric | FY 2025 (Jan 26, 2025) | FY 2026 (Jan 25, 2026) | Change ($) | % Change |
| :--- | :--- | :--- | :--- | :--- |
| **Research and Development** | $12,914 million | $18,497 million | $5,583 million | 43% |
| **Sales, General and Administrative** | $3,491 million | $4,579 million | $1,088 million | 31% |
| **Total Operating Expenses** | $16,405 million | $23,076 million | $6,671 million | 41% |

#### Percentage Change Calculations
*   **Research and Development:** $((18,497 - 12,914) / 12,914) \times 100 = 43.2\%$ (Reported as 43% in context)
*   **Sales, General and Administrative:** $((4,579 - 3,491) / 3,491) \times 100 = 31.2\%$ (Reported as 31% in context)
*   **Total Operating Expenses:** $((23,076 - 16,405) / 16,405) \times 100 = 40.7\%$ (Reported as 41% in context)

### In-Depth Analysis and Trends

**1. Research and Development (R&D) Trends**
*   **Absolute Growth:** R&D expenses increased from **$12,914 million** in FY 2025 to **$18,497 million** in FY 2026, representing a year-over-year increase of **$5,583 million**.
*   **Relative Growth:** The **43%** increase in R&D spending indicates a significant acceleration in investment relative to the prior year.
*   **Operational Leverage:** Despite the substantial increase in absolute R&D dollars, R&D as a percentage of revenue declined from **9.9%** in FY 2025 to **8.6%** in FY 2026. This suggests that revenue growth (which increased by **65%** to **$215,938 million**) is outpacing the growth in R&D costs, demonstrating positive operating leverage. The context notes that the revenue increase was driven by major platform shifts, specifically accelerated computing and AI, with Data Center computing revenue growing **59%** driven by demand for the Blackwell computing platform.

**2. Sales, General and Administrative (SG&A) Trends**
*   **Absolute Growth:** SG&A expenses increased from **$3,491 million** in FY 2025 to **$4,579 million** in FY 2026, a rise of **$1,088 million**.
*   **Relative Growth:** SG&A grew by **31%**, which is slower than the **65%** growth in total revenue.
*   **Operational Leverage:** Similar to R&D, SG&A efficiency improved. SG&A as a percentage of revenue decreased from **2.7%** in FY 2025 to **2.1%** in FY 2026. This indicates that the company is scaling its sales and administrative functions more efficiently than its top-line growth.

**3. Total Operating Expenses**
*   **Aggregate Trend:** Total operating expenses rose from **$16,405 million** in FY 2025 to **$23,076 million** in FY 2026, an increase of **41%**.
*   **Margin Impact:** Total operating expenses as a percentage of revenue decreased from **12.6%** in FY 2025 to **10.7%** in FY 2026. This **1.9 percentage point** reduction in the expense ratio directly contributed to the expansion in operating margins.
*   **Operating Income Correlation:** The efficiency in managing operating expenses, combined with revenue growth, resulted in Operating Income increasing from **$81,453 million** in FY 2025 to **$130,387 million** in FY 2026 (an increase of **60%**). Operating income as a percentage of revenue decreased slightly from **62.4%** to **60.4%**, primarily due to the larger decline in Gross Margin (from **75.0%** to **71.1%**) which offset the improvements in operating expense ratios.

**Conclusion**
NVIDIA demonstrated strong operational discipline in FY 2026. While both R&D and SG&A expenses increased in absolute terms to support business growth, they grew at a slower rate than revenue. This resulted in a lower operating expense ratio (10.7% vs 12.6%) and supported a **60%** increase in Operating Income. The heavy investment in R&D (up 43%) aligns with the strategic focus on AI and accelerated computing platforms, such as Blackwell, which drove the majority of the revenue growth.

## Sources
- Nvidia.pdf | Page 105
- Nvidia.pdf | Page 71
- Nvidia.pdf | Page 57
- Nvidia.pdf | Page 58
- Nvidia.pdf | Page 104
- Nvidia.pdf | Page 53
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 75

In [27]:
query = """Evaluate NVIDIA's overall corporate strategy and capital allocation decisions.

Focus on:
- Core business model and diversification efforts
- R&D investment trends
- Mergers & acquisitions strategy
- Share buyback and dividend policy
- Long-term vision in AI, robotics, autonomous vehicles, and Omniverse
- Management's capital allocation priorities

Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors 

Provide investor implications."""

response = financial_rag_with_bleu(query = query,company = "Nvidia")

display(Markdown(response))



🔍 Company: Nvidia | Query: Evaluate NVIDIA's overall corporate strategy and capital all...
   Retrieval Quality: 11/18 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 11/18 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7674 | Recall: 0.7715 | F1: 0.7694


# Financial Analysis — Nvidia

**Financial Analysis:**

**Type of Question:** B (Strategy, Technology, and Non-numeric Topic)
*Note: The provided Context does not contain specific financial figures for Revenue, Operating Expenses, Net Income, R&D expenses, M&A costs, Share Buybacks, or Dividends. Therefore, per the Absolute Rules, specific financial metrics for these categories are "not disclosed in the retrieved sections." The analysis below is strictly based on the strategic, operational, and qualitative information present in the Context.*

### 1. Core Business Model and Diversification Efforts
NVIDIA has transitioned from a company originally focused on PC graphics to a "data center scale AI infrastructure company reshaping all industries." The core business model is built on accelerated computing, leveraging the foundational NVIDIA CUDA development platform that runs on all NVIDIA GPUs.

*   **Diversification:** The company has expanded into several computationally intensive fields, including scientific computing, AI, data science, autonomous vehicles, robotics, and digital twin applications.
*   **Segmentation:** NVIDIA operates through two primary segments:
    1.  **Compute & Networking:** Includes Data Center accelerated computing and networking platforms, AI solutions and software, and Automotive platforms (autonomous and electric vehicle solutions).
    2.  **Graphics:** Includes GeForce GPUs for gaming and PCs, and Quadro/NVIDIA RTX GPUs for enterprise workstation graphics.
*   **Strategic Shift:** The context highlights that revenue growth in fiscal year 2026 was driven by data center compute and networking platforms for accelerated computing and AI solutions, indicating a strategic pivot away from pure gaming hardware toward enterprise and data center infrastructure.

### 2. R&D Investment Trends
*   **Status:** Specific R&D expense figures are **not disclosed in the retrieved sections.**
*   **Qualitative Insight:** The context indicates a strong commitment to technology development through "extreme co-design" in the Blackwell architecture, where chips, networking, systems, software, and algorithms are holistically architected. This suggests high investment in integrated system design rather than isolated component development.

### 3. Mergers & Acquisitions Strategy
*   **Status:** Specific M&A transaction values or counts are **not disclosed in the retrieved sections.**
*   **Accounting Policy:** The context details NVIDIA’s accounting approach for business combinations. The company applies a "screen test" to determine if a transaction is an asset acquisition or a business combination. If a business combination is identified, the fair value of the purchase price is allocated to tangible assets, liabilities, and intangible assets, with the excess recorded as goodwill.
*   **Strategic Implication:** The presence of detailed accounting policies for business combinations and intangible asset amortization implies that M&A is a considered part of the corporate strategy, likely used to acquire technology or talent that complements the accelerated computing roadmap.

### 4. Share Buyback and Dividend Policy
*   **Status:** Information regarding share buybacks and dividend policies is **not disclosed in the retrieved sections.**
*   **Capital Allocation Context:** Instead of returning capital to shareholders via buybacks or dividends (which are not mentioned), the context highlights significant capital deployment into external investments and infrastructure guarantees (see Section 6).

### 5. Long-term Vision in AI, Robotics, Autonomous Vehicles, and Omniverse
*   **AI Vision:** NVIDIA positions itself as a data center scale AI infrastructure company. The Blackwell architecture is central to this vision, featuring "extreme co-design" to maximize performance and scale. The company notes that hundreds of thousands of GPUs can be interconnected to function as a single giant computer, facilitating large-scale AI model training and inference.
*   **Robotics and Autonomous Vehicles:** The Compute & Networking segment explicitly includes "Automotive platforms and autonomous and electric vehicle solutions including software." The company has leveraged its GPU architecture to create platforms for robotics and autonomous vehicles, indicating a long-term bet on physical AI and mobility.
*   **Omniverse:** While the term "Omniverse" is not explicitly defined in the text, the context mentions "digital twin applications" and "3D graphics" as key expansion areas. The technology stack includes domain-specific software libraries and SDKs for these workloads, suggesting that digital twin capabilities are a core part of the long-term software strategy.

### 6. Management's Capital Allocation Priorities
Management’s capital allocation is heavily skewed toward supporting the AI ecosystem and infrastructure buildout, rather than shareholder returns.

*   **Private Equity and Infrastructure Investments:** In fiscal year 2026, NVIDIA invested **$17.5 billion** in private companies and infrastructure funds, primarily to support early-stage startups. These investments include AI model makers that purchase NVIDIA products directly or through Cloud Service Providers (CSPs).
*   **Infrastructure Guarantees:** NVIDIA provided **$3.5 billion** in land, power, and shell guarantees to early-stage companies to support the build-out of complex data center infrastructures. These guarantees are generally over multi-year periods.
*   **Risk Acknowledgment:** Management acknowledges that these investments are illiquid and non-marketable, and there is no assurance of return. Additionally, the company faces risks related to the availability of data centers, energy, and capital for its customers. Expanding energy capacity is described as a "complex, multi-year process" with significant regulatory and technical challenges.

### Investor Implications and Advice

Based on the strategic and capital allocation data provided in the context, the following implications and advice are offered to investors:

1.  **Focus on Data Center Dominance:** Investors should view NVIDIA not as a gaming hardware company, but as a critical infrastructure provider for the AI era. The fact that Blackwell architectures represented the majority of Data Center revenue in fiscal year 2026 underscores the importance of the data center segment. Investors should monitor the adoption rate of Blackwell and subsequent architectures.
2.  **Assess Ecosystem Lock-in:** NVIDIA’s strategy of investing **$17.5 billion** in private companies and providing **$3.5 billion** in infrastructure guarantees is a defensive and offensive move to secure its customer base. By funding the startups and AI model makers that use its chips, NVIDIA is creating a closed ecosystem. Investors should evaluate whether this "vendor financing" model creates sustainable demand or introduces credit risk if these startups fail.
3.  **Monitor Infrastructure Bottlenecks:** The context explicitly warns that the availability of data centers, energy, and capital is crucial. Any shortage in these resources could impact future revenue. Investors should watch for signs of energy constraints or regulatory hurdles in data center construction, as these are identified as key risks to financial performance.
4.  **Regulatory Risk:** NVIDIA faces increased scrutiny from regulators worldwide (EU, US, UK, China, South Korea, etc.) regarding antitrust, competition, and cybersecurity. The French Competition Authority’s inquiry into the graphics card and CSP market is a specific example. Investors should consider the potential for increased compliance costs or operational restrictions due to regulatory actions.
5.  **Lack of Shareholder Returns:** The absence of disclosed buyback or dividend policies, combined with massive capital outflows for investments and guarantees, suggests that NVIDIA is in a high-growth, capital-intensive phase. Investors should not expect significant cash returns from the company in the near term and should instead focus on capital appreciation driven by revenue growth in the AI infrastructure space.
6.  **Diversification Risk:** While NVIDIA has diversified into automotive, robotics, and digital twins, the context emphasizes that revenue growth is driven by data center compute. Investors should be cautious about overvaluing the company based on non-core segments (like gaming or automotive) if the data center segment faces headwinds.

**Conclusion:**
NVIDIA’s strategy is aggressively focused on becoming the indispensable infrastructure layer for AI. Capital is being allocated to secure market share through direct investments in customers and partners, rather than returned to shareholders. Investors should align their expectations with a high-growth, high-capital-expenditure model, while closely monitoring regulatory risks and infrastructure constraints that could impede the buildout of AI capabilities.

## Sources
- Nvidia.pdf | Page 37
- Nvidia.pdf | Page 52
- Nvidia.pdf | Page 81
- Nvidia.pdf | Page 104
- Nvidia.pdf | Page 4
- Nvidia.pdf | Page 83
- Nvidia.pdf | Page 53

In [28]:
query = """Provide a comprehensive financial analysis of Tesla using the latest SEC 10-k filings.

Focus on:
- Total revenue breakdown in 2025 vs 2024 vs 2023 and year-over-year growth trends (Automotive vs Energy Generation & Storage)
- Discuss Automotive segment performance 
- Discuss Energy Generation and Storage segment growth and margins
- Gross margin trends and key drivers
- Operating income, net income, and profitability trends
- Free cash flow generation and capital expenditure
- Overall financial health and liquidity position


Explain and discuss each above points in depth and not just overview 
Adice investors what they could do based on the discussion that is advised for the investors."""


response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Provide a comprehensive financial analysis of Tesla using th...
   Retrieval Quality: 37/24 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 37/24 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7565 | Recall: 0.7737 | F1: 0.7650


# Financial Analysis — Tesla

# Financial Analysis: Tesla, Inc. (FY2023–FY2025)

## 1. Executive Summary
Tesla’s financial performance for the fiscal year ended December 31, 2025, reflects a period of significant margin compression and declining profitability despite growth in its Energy Generation and Storage segment. Total revenues decreased year-over-year, driven primarily by a sharp decline in Automotive sales revenue. While the Energy segment showed robust growth, it was insufficient to offset the automotive downturn. Operating income and net income declined substantially compared to 2024, and liquidity remains stable but shows a slight decrease in total cash and restricted cash compared to the prior year.

## 2. Revenue Breakdown and Growth Trends

The following table details Tesla’s revenue composition for the last three fiscal years.

| Metric | FY 2023 ($ millions) | FY 2024 ($ millions) | FY 2025 ($ millions) | YoY Change 2024→2025 |
| :--- | :--- | :--- | :--- | :--- |
| **Automotive Sales** | $78,509 | $72,480 | $65,821 | Decreased |
| **Automotive Regulatory Credits** | $1,790 | $2,763 | $1,993 | Decreased |
| **Automotive Leasing** | $2,120 | $1,827 | $1,712 | Decreased |
| **Total Automotive Revenues** | $82,419 | $77,070 | $69,526 | Decreased |
| **Energy Generation and Storage** | $6,035 | $10,086 | $12,771 | Increased |
| **Services and Other** | $8,319 | $10,534 | $12,530 | Increased |
| **Total Revenues** | $96,773 | $97,690 | $94,827 | Decreased |

### Automotive Segment Performance
*   **Revenue Trend:** Total Automotive Revenues decreased from $77,070 million in FY 2024 to $69,526 million in FY 2025.
*   **Drivers:** The decline is primarily attributed to a decrease in deliveries and lower average selling prices. Specifically, Automotive Sales revenue fell from $72,480 million to $65,821 million.
*   **Regulatory Credits:** Automotive regulatory credits decreased from $2,763 million in FY 2024 to $1,993 million in FY 2025.
*   **Leasing:** Automotive leasing revenue declined from $1,827 million to $1,712 million.

### Energy Generation and Storage Segment Growth
*   **Revenue Trend:** Energy Generation and Storage revenue increased from $10,086 million in FY 2024 to $12,771 million in FY 2025.
*   **Growth Calculation:** The increase is $2,685 million ($12,771 - $10,086).
    *   Percentage Change: $((12,771 - 10,086) / 10,086) \times 100 = 26.6\%$ (Context states "increased $2.69 billion, or 27%").
*   **Drivers:** Growth was driven by increases in Megapack and Powerwall deployments, partially offset by a decrease in the average selling price of Megapack.

### Services and Other Segment
*   **Revenue Trend:** Services and other revenue increased from $10,534 million in FY 2024 to $12,530 million in FY 2025.
*   **Growth Calculation:** The increase is $1,996 million ($12,530 - $10,534).
    *   Percentage Change: $((12,530 - 10,534) / 10,534) \times 100 = 18.9\%$ (Context states "increased $2.00 billion, or 19%").
*   **Drivers:** Increases in paid Supercharging sessions, non-warranty maintenance services, collision revenue, used vehicle sales volume, and automotive insurance business revenue.

## 3. Gross Margin Trends and Key Drivers

The following table details Gross Profit and Margin trends.

| Metric | FY 2024 | FY 2025 | Change |
| :--- | :--- | :--- | :--- |
| **Gross Profit** | $17,450 million | $17,094 million | Decreased |
| **Total Automotive Gross Margin** | 18.4% | 17.8% | Decreased |
| **Total Automotive & Services Gross Margin** | 16.9% | 16.2% | Decreased |

### Analysis of Drivers
*   **Automotive Gross Margin:** Decreased from 18.4% to 17.8%. The context attributes this primarily to a decrease in regulatory credits revenue and changes in automotive sales revenue and cost of automotive sales revenue.
*   **Cost of Automotive Sales:** Decreased by $5.60 billion (9%) in FY 2025 compared to FY 2024. This decrease was due to lower deliveries and lower average cost per unit (sales mix and lower material costs), partially offset by lower fixed cost absorption and an increase in tariffs.
*   **Cost of Services and Other:** Increased by $1.68 billion (17%) in FY 2025 compared to FY 2024, driven by higher volumes in used vehicle sales, Supercharging, insurance, and maintenance.
*   **Combined Margin:** The gross margin for the total automotive & services and other segment decreased from 16.9% to 16.2%, driven by the drop in regulatory credits, partially offset by improved services margins.

## 4. Operating Income, Net Income, and Profitability Trends

The following table details operating and net income figures.

| Metric | FY 2023 ($ millions) | FY 2024 ($ millions) | FY 2025 ($ millions) | YoY Change 2024→2025 |
| :--- | :--- | :--- | :--- | :--- |
| **Total Operating Expenses** | $8,769 | $10,374 | $12,739 | Increased |
| **Income from Operations** | $8,891 | $7,076 | $4,355 | Decreased |
| **Net Income** | $14,974 | $7,153 | $3,855 | Decreased |
| **Net Income Attributable to Common Stockholders** | $14,997 | $7,091 | $3,794 | Decreased |
| **Diluted EPS** | $4.30 | $2.04 | $1.08 | Decreased |

### Operating Expense Analysis
*   **Total Operating Expenses:** Increased from $10,374 million in FY 2024 to $12,739 million in FY 2025.
*   **Research and Development:** Increased from $4,

## Sources
- Tesla.pdf | Page 54
- Tesla.pdf | Page 66
- Tesla.pdf | Page 64
- Tesla.pdf | Page 77
- Tesla.pdf | Page 74
- Tesla.pdf | Page 99
- Tesla.pdf | Page 82

In [29]:
query = "Discuss about operating expenses of Tesla in depth"

"Quote exact figures from given retrived context you received and compare correct and accurate trends where available."

response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Discuss about operating expenses of Tesla in depth
   Retrieval Quality: 3/2 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 3/2 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7471 | Recall: 0.8433 | F1: 0.7923


# Financial Analysis — Tesla

### Financial Analysis: Tesla Operating Expenses

Based on the provided financial statements and management discussion, the following analysis details Tesla’s operating expenses for the fiscal years ended December 31, 2023, 2024, and 2025.

#### Key Operating Expense Metrics

| Metric | 2023 | 2024 | 2025 | Change (2025 vs 2024) | % Change (2025 vs 2024) |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **Research and Development** | $3,969 million | $4,540 million | $6,411 million | +$1,871 million | +41.2% |
| **Selling, General and Administrative** | $4,800 million | $5,150 million | $5,834 million | +$684 million | +13.3% |
| **Restructuring and other** | $0 million | $684 million | $494 million | -$190 million | -27.8% |
| **Total Operating Expenses** | $8,769 million | $10,374 million | $12,739 million | +$2,365 million | +22.8% |

*Note: Percentages calculated as ((Later Year - Earlier Year) / Earlier Year) × 100.*

#### Detailed Component Analysis

**1. Research and Development (R&D)**
*   **Trend:** R&D expenses increased significantly, rising from $4,540 million in 2024 to $6,411 million in 2025.
*   **Drivers:** The increase of $1,871 million (approximately 41%) was primarily driven by:
    *   Higher costs related to AI and other programs as Tesla expands its product roadmap and technologies.
    *   An increase in stock-based compensation of $500 million.
*   **Revenue Ratio:** R&D expense as a percentage of revenue increased from 5% in 2024 to 7% in 2025. This rise is attributed to the higher absolute R&D spend combined with a decrease in total revenue year-over-year.

**2. Selling, General and Administrative (SG&A)**
*   **Trend:** SG&A expenses increased from $5,150 million in 2024 to $5,834 million in 2025.
*   **Drivers:** The increase of $684 million (13%) was driven by:
    *   A $354 million increase in operating expenses, including legal charges.
    *   A $256 million increase in employee and labor costs, including professional services.
    *   A $235 million increase in stock-based compensation.
    *   *Offsets:* These increases were partially offset by an $83 million decrease in marketing expenses and a $78 million decrease in facilities-related expenses.
*   **Revenue Ratio:** SG&A as a percentage of revenues increased from 5% in 2024 to 6% in 2025.

**3. Restructuring and Other**
*   **Trend:** This line item decreased from $684 million in 2024 to $494 million in 2025.
*   **Context:** In 2023, this expense was not disclosed (represented as "—" in the table), indicating no material restructuring charges were recorded for that year.

#### Impact on Operating Income
The aggregate increase in Total Operating Expenses from $10,374 million in 2024 to $12,739 million in 2025 contributed to a decline in Income from operations. Despite a slight increase in Gross Profit (from $17,450 million in 2024 to $17,660 million in 2025), the substantial rise in operating costs resulted in Income from operations decreasing from $7,076 million in 2024 to $4,355 million in 2025.

#### Strategic Context
The significant escalation in R&D and SG&A expenses aligns with Tesla’s strategic focus on AI initiatives, compute infrastructure, and data centers. Management notes that capital expenditures are expected to exceed $20 billion in 2026, driven by these AI initiatives and the expansion of manufacturing and R&D production lines. This suggests that the current increase in operating expenses is a deliberate investment in future technological capabilities and fleet expansion.

## Sources
- Tesla.pdf | Page 120
- Tesla.pdf | Page 82
- Tesla.pdf | Page 66
- Tesla.pdf | Page 68
- Tesla.pdf | Page 72

In [30]:
query = """Analyze Tesla's autonomous driving technology, Full Self-Driving (FSD), and Robotaxi strategy in detail.

Focus on:
- Current status and capabilities of FSD (Supervised)
- Progress toward unsupervised autonomy and regulatory approvals
- Robotaxi business model and deployment plans
- Competitive advantages in AI and neural networks
- Timeline and risks for commercialization
- Management's vision and expected financial impact



Explain each point above in very much depth and with intuition

Quote specific statements from the filings and conclude with investor implications."""

response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Analyze Tesla's autonomous driving technology, Full Self-Dri...
   Retrieval Quality: 16/19 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 16/19 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7723 | Recall: 0.7848 | F1: 0.7785


# Financial Analysis — Tesla

**Financial Analysis: Tesla Autonomous Driving & Robotaxi Strategy**

**Question Type:** B (Architecture, Technology, Strategy, Risks)
*Note: As this question pertains to strategy, technology, and qualitative outlook rather than specific financial line items (Revenue, Net Income, etc.), a financial data table is not applicable. The analysis below is derived strictly from the provided Context.*

### 1. Current Status and Capabilities of FSD (Supervised)
Tesla’s Full Self-Driving (FSD) technology is currently deployed under a "Supervised" framework, indicating that human oversight remains a mandatory component of the system. The filing explicitly notes that the term "FSD (Supervised)" includes the equivalent naming convention "FSD (Capability)" used in the European, Middle East, and Asia-Pacific regions, suggesting a global standardization of the feature set despite regional naming differences.

The company is actively leveraging developments in its proprietary FSD features, battery cell technologies, and other proprietary technologies. The strategy involves continuously improving and deploying these capabilities, with a specific focus on "future autonomous capabilities." The current operational status relies on existing vehicle platforms, specifically the Model Y, which serves as the primary hardware for the current Robotaxi deployment.

> **Quote:** "We have planned electric vehicles to address additional vehicle markets, and will continue leveraging developments in our proprietary Full Self-Driving (“FSD”) (Supervised) features, battery cell and other technologies."

### 2. Progress Toward Unsupervised Autonomy and Regulatory Approvals
The provided context does not disclose specific regulatory approval milestones or a confirmed timeline for unsupervised (Level 4/5) autonomy. Instead, the filing highlights a significant legal risk regarding the effectiveness of current autonomous features. A proposed class action filed on August 4, 2025, in the U.S. District Court Western District of Texas alleges material misrepresentations in public filings regarding the effectiveness of Autopilot, FSD (Supervised), and Robotaxi.

This litigation covers stock purchases between April 19, 2023, and June 22, 2025. The presence of this lawsuit suggests that the gap between public expectations of autonomous capability and the current "Supervised" reality is a material risk factor. The company states it intends to "vigorously defend" itself but cannot predict the outcome or impact, indicating that regulatory and legal hurdles remain unresolved and potentially costly.

> **Quote:** "The complaint alleges that the defendants violated federal securities laws through alleged material misrepresentations in public filings regarding the effectiveness of Autopilot, Full-Self Driving (Supervised), and Robotaxi."

### 3. Robotaxi Business Model and Deployment Plans
Tesla launched its Robotaxi service in June 2025, defining it as an "autonomous ride-hailing platform." The business model is designed to be service-driven, leveraging AI, software, and fleet-based profits. The current deployment utilizes Model Y vehicles, but the long-term strategy involves the "Cybercab," a purpose-built autonomous vehicle.

The company expects the Robotaxi service to open access to an expanded customer base as transportation modes evolve. The deployment is described as being "refined" and "expanded" post-launch, capitalizing on AI investments and scalable mobility infrastructure. The transition from using existing consumer vehicles (Model Y) to purpose-built hardware (Cybercab) is a critical phase in the commercialization timeline, with plans to "mass produce" the Cybercab.

> **Quote:** "In June 2025, we launched our Robotaxi service, an autonomous ride-hailing platform that harnesses our technology and vehicles... Our Robotaxi business currently operates with Model Y vehicles but, in time, will include Cybercab, our purpose-built autonomous vehicle."

### 4. Competitive Advantages in AI and Neural Networks
Tesla positions its competitive advantage in the autonomous solutions market through its proprietary AI and neural network capabilities. The company aims to become a "top provider of autonomous solutions" by competing against both traditional ride-hailing/taxi services and emerging AI/robotics competitors.

Key differentiators cited include:
*   **Proprietary Technology:** Continued progress on FSD (Supervised) and neural network capabilities.
*   **Infrastructure Integration:** Leverage of the Supercharger network and infotainment offerings.
*   **AI Integration:** The broader corporate focus is on bringing "artificial intelligence (“AI”) into the real world," extending beyond vehicles to include AI robots ("Bots") such as Optimus.
*   **Vertical Integration:** The strategy includes vertically integrating and localizing the supply chain, which supports the scalability of the Robotaxi fleet.

> **Quote:** "We expect our Robotaxi service to compete in this developing market, along with traditional ride-hailing and taxi services, through continued progress on our FSD (Supervised) and neural network capabilities, Supercharger network and infotainment offerings."

### 5. Timeline and Risks for Commercialization
The commercialization timeline is tied to the June 2025 launch of the Robotaxi service and the subsequent mass production of the Cybercab. However, the filing outlines significant risks that could hinder this timeline:

*   **Market Adoption Risk:** Success depends on "acceptance and adoption by consumers of autonomous driving solutions." If uptake rates do not meet expectations, business prospects may be harmed.
*   **Competitive Pressure:** The market is described as "highly competitive," with both established and emerging companies introducing similar products.
*   **Operational Scope:** The company acknowledges that if it cannot maintain operations at a scope commensurate with global market conditions, or if it must suspend operations, its financial condition may be harmed.
*   **Legal Risk:** The ongoing class action regarding misrepresentations of FSD/Robotaxi effectiveness poses a direct threat to financial stability and reputation.

> **Quote:** "If the uptake rate for autonomous driving solutions does not develop as we expect, our business, prospects, financial condition and operating results may be harmed."

### 6. Management's Vision and Expected Financial Impact
Management’s vision is to transition from a pure vehicle manufacturer to a service-driven entity. The core objective is to "unlock the potential to advance a service-driven business model based on AI, software and fleet-based profits." This implies a shift in revenue recognition from one-time vehicle sales to recurring revenue streams from FSD subscriptions and Robotaxi rides.

The financial impact is expected to be driven by:
*   **Profitable Growth:** Focused on a "differentiated and efficiently managed product portfolio."
*   **Cost Reduction:** Continuous efforts to reduce manufacturing costs and lower the cost of ownership for customers.
*   **Scale:** Increasing vehicle production, utilized capacity, and delivery capabilities.

The filing notes that in 2025, the company produced approximately 1.66 million consumer vehicles and delivered approximately 1.64 million, providing the baseline fleet from which the Robotaxi service is currently operating. The long-term financial impact hinges on the successful scaling of the Robotaxi fleet and the monetization of FSD subscriptions.

> **Quote:** "We expect this service will open access to an expanded customer base as modes of transportation evolve, and, along with products such as FSD (Supervised) subscriptions, unlock the potential to advance a service-driven business model based on AI, software and fleet-based profits."

### Investor Implications

1.  **Revenue Model Shift:** Investors should monitor the transition from hardware-centric revenue to service-centric revenue (FSD subscriptions and Robotaxi ride-hailing). The success of this pivot is critical for long-term valuation multiples, as service revenue typically commands higher margins and stickiness.
2.  **Legal Overhang:** The class action filed in August 2025 regarding "material misrepresentations" of FSD/Robotaxi effectiveness is a material risk. Investors must assess the potential for monetary damages and reputational harm, which could impact stock price volatility and future capital raising capabilities.
3.  **Execution Risk on Cybercab:** The current Robotaxi service

## Sources
- Tesla.pdf | Page 6
- Tesla.pdf | Page 140
- Tesla.pdf | Page 20
- Tesla.pdf | Page 54
- Tesla.pdf | Page 28

In [31]:
query = """Provide a detailed analysis of Tesla's Energy Generation and Storage segment.

Focus on:
- Powerwall and Megapack products and their applications
- Revenue growth, deployment volumes, and profitability trends
- Virtual power plants and software platforms (Powerhub, Autobidder)
- Competitive positioning vs traditional utilities and other energy storage companies
- Future growth outlook and strategic importance to Tesla

Explain each point above in very much depth and with intuition

Use exact figures and management commentary from the filings."""

response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Provide a detailed analysis of Tesla's Energy Generation and...
   Retrieval Quality: 23/18 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 23/18 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7700 | Recall: 0.8400 | F1: 0.8035


# Financial Analysis — Tesla

**Financial Analysis: Tesla Energy Generation and Storage Segment**

**1. Product Architecture and Applications**
Tesla’s Energy Generation and Storage segment is built upon two primary lithium-ion battery energy storage products: **Powerwall** and **Megapack**.

*   **Powerwall:** This product is designed for residential and small commercial facilities. Tesla distributes Powerwall through direct sales, leasing to customers, and channel partners. Its primary function is to store energy for these smaller-scale applications.
*   **Megapack:** This is an energy storage solution targeted at commercial, industrial, utility, and energy generation customers. Megapack units can be grouped together to form larger installations with capacities of gigawatt hours (“GWh”) or greater.
*   **Strategic Utility of Megapack:** The filing highlights that as AI infrastructure drives rapid load growth, Megapack helps increase the utilization of existing generation and transmission capacity. This results in a more efficient use of the electric grid, positioning the product not just as a storage device, but as a critical infrastructure tool for grid stability and efficiency.
*   **Energy Generation Offerings:** The segment also includes energy generation systems sold directly or through channel partners. Tesla designs and manufactures certain components, including solar panels and **Solar Roof** (which combines premium glass roof tiles with energy generation). Notably, Tesla began manufacturing a new residential retrofit solar panel in 2025, with initial customer deliveries commencing in January 2026. Both Solar Roof and the new retrofit panel are designed to integrate with Powerwall.

**2. Financial Performance: Revenue, Costs, and Profitability**
The segment demonstrated significant top-line growth and margin expansion in the year ended December 31, 2025, compared to the year ended December 31, 2024.

| Metric | Year Ended Dec 31, 2024 | Year Ended Dec 31, 2025 | Change ($) | Change (%) |
| :--- | :--- | :--- | :--- | :--- |
| **Energy Generation and Storage Revenue** | $9.96 billion* | $12.65 billion* | $2.69 billion | 27% |
| **Cost of Energy Generation and Storage Revenue** | $7.35 billion* | $8.87 billion* | $1.52 billion | 20% |
| **Gross Margin** | 26.2% | 29.8% | +3.6 pts | N/A |

*\*Note: The absolute dollar values for 2024 and 2025 Revenue and Cost of Revenue are derived from the stated changes and percentages in the context. Specifically, Revenue increased by $2.69 billion (27%), and Cost of Revenue increased by $1.52 billion (20%). The context explicitly states the change amounts and percentages, as well as the Gross Margin figures.*

*   **Revenue Growth:** Energy generation and storage revenue increased **$2.69 billion**, or **27%**, in 2025 compared to 2024.
    *   *Drivers:* This growth was primarily due to increases in **Megapack** and **Powerwall** deployments.
    *   *Offsetting Factor:* The revenue growth was partially offset by a decrease in the average selling price of Megapack.
*   **Cost Structure:** Cost of energy generation and storage revenue increased **$1.52 billion**, or **20%**, in 2025 compared to 2024.
    *   *Drivers:* Costs rose primarily from the same increase in Megapack and Powerwall deployments.
    *   *Efficiency Gains:* The cost increase was partially offset by a decrease in the average cost per unit for both Megapack and Powerwall. This reduction was driven by lower raw material costs and lower manufacturing costs for Megapack, in part from the ramp of the **Shanghai Megafactory**.
    *   *Headwinds:* These efficiencies were partially offset by higher tariffs.
*   **Profitability:** Gross margin for the energy generation and storage segment increased from **26.2%** in 2024 to **29.8%** in 2025. This expansion was primarily due to the changes in revenue and cost of revenue discussed above, indicating that the segment is becoming more profitable on a per-unit basis despite volume increases and tariff pressures.

**3. Software Platforms and Virtual Power Plants**
Tesla leverages software capabilities to remotely control and dispatch its energy storage systems, enhancing their value proposition beyond simple hardware sales.

*   **Autobidder:** This is a real-time energy control and optimization platform specifically for **Megapack** batteries. It allows for sophisticated management of large-scale storage assets.
*   **Powerhub:** This platform is designed for distributed energy resources, including **Powerwall**-enabled virtual power plants. It enables the aggregation and optimization of smaller residential storage units.
*   **AI Integration:** Tesla leverages its capabilities in AI to ensure that every energy storage product can be enhanced through firmware updates and optimized by these software platforms. This creates a recurring value proposition and potential for future software-related revenue or service enhancements.

**4. Competitive Positioning and Strategic Importance**
*   **Grid Efficiency vs. Traditional Utilities:** The filing explicitly states that Megapack helps increase the utilization of existing generation and transmission capacity, resulting in a more efficient use of the electric grid. This positions Tesla as a solution provider that enhances grid reliability and efficiency, particularly in the context of rapid load growth driven by AI infrastructure. This is a strategic advantage over traditional utilities that may struggle with peak demand management.
*   **Manufacturing Scale and Cost Leadership:** The ramp of the **Shanghai Megafactory** contributed to lower manufacturing costs for Megapack. This suggests Tesla is leveraging its automotive manufacturing expertise to achieve cost leadership in energy storage, allowing it to offer competitive pricing (even with decreasing average selling prices) while expanding margins.
*   **Integrated Ecosystem:** By integrating solar generation (Solar Roof, retrofit panels) with storage (Powerwall, Megapack) and software (Powerhub, Autobidder), Tesla offers a comprehensive energy solution. This integrated approach differentiates Tesla from pure-play battery manufacturers or traditional solar installers.
*   **Strategic Importance:** The segment is one of Tesla’s two reportable segments (alongside Automotive). The 27% revenue growth and 3.6 percentage point increase in gross margin highlight its increasing strategic importance and contribution to Tesla’s overall financial performance. The ability to leverage component-level technologies from vehicles for energy storage products further underscores the synergies within Tesla’s business model.

**5. Future Growth Outlook**
*   **Deployment Momentum:** The primary driver of revenue growth in 2025 was increased deployments of Megapack and Powerwall. This suggests continued demand for energy storage solutions.
*   **New Product Launches:** The commencement of initial customer deliveries for the new residential retrofit solar panel in January 2026 indicates a pipeline of new products that could drive further growth in the energy generation side of the segment.
*   **AI-Driven Demand:** The mention of AI infrastructure driving rapid load growth implies a structural tailwind for Megapack demand, as data centers and AI workloads require reliable and efficient power management.
*   **Software Monetization Potential:** The continued development of software capabilities for remote control and dispatch (Autobidder, Powerhub) suggests potential for future revenue streams from software services or enhanced service contracts, adding a higher-margin component to the segment’s revenue mix.

In summary, Tesla’s Energy Generation and Storage segment is experiencing strong growth in both revenue and profitability, driven by increased deployments of Megapack and Powerwall, cost efficiencies from manufacturing scale (particularly Shanghai), and the strategic value of its software platforms. The segment is well-positioned to benefit from the growing demand for grid stability and efficiency in the era of AI-driven load growth.

## Sources
- Tesla.pdf | Page 7
- Tesla.pdf | Page 64
- Tesla.pdf | Page 9
- Tesla.pdf | Page 145
- Tesla.pdf | Page 66

In [32]:
query = """Analyze Tesla's supply chain, manufacturing operations, and associated risks.

Focus on:
- Dependency on key suppliers (especially battery cells)
- Manufacturing facilities and production ramp plans
- Geopolitical and raw material risks (lithium, nickel, etc.)
- Vertical integration efforts and in-house battery cell production
- Impact of potential disruptions on production and financials

Explain each point above in very much depth and with intuition


Assess overall supply chain vulnerability and mitigation strategies."""

response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))




🔍 Company: Tesla | Query: Analyze Tesla's supply chain, manufacturing operations, and ...
   Retrieval Quality: 19/17 words matched
📝 Pass 1: Generating Response
   Retrieval Quality: 19/17 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.8091 | Recall: 0.8142 | F1: 0.8116


# Financial Analysis — Tesla

**Financial Analysis: Tesla Supply Chain, Manufacturing Operations, and Risk Assessment**

Based on the retrieved sections of Tesla’s financial filings, the following analysis details the company’s manufacturing footprint, supply chain dependencies, and associated operational risks. As this inquiry focuses on operational architecture and qualitative risk factors rather than specific financial line items (Revenue, Operating Income, etc.), no financial table is presented. All insights are derived strictly from the provided text.

### 1. Manufacturing Footprint and Production Strategy

Tesla maintains a global manufacturing network designed to balance local market affordability with global scale efficiencies.

*   **U.S. Facilities:** Operations are concentrated in California, New York, Texas, and Nevada. These facilities handle the manufacturing and assembly of vehicles, battery packs, battery cells, energy storage components, and solar products.
*   **International Facilities:** Manufacturing is also located in China and Germany. The strategic rationale for these locations is to increase vehicle affordability for local customers by reducing transportation and manufacturing costs while mitigating the impact of unfavorable tariffs.
*   **Capacity Expansion:** The company states it continues to expand production capacity at existing facilities. The strategy involves increasing cost-competitiveness in significant markets by strategically adding local manufacturing, often through partnerships with suppliers.

### 2. Supply Chain Structure and Supplier Dependency

Tesla’s supply chain is characterized by a mix of broad sourcing and critical single-source dependencies, particularly regarding battery technology.

*   **Broad Sourcing:** Products utilize parts sourced from thousands of suppliers globally. Close relationships have been developed with key partners supplying battery cells, electronics, and complex vehicle assemblies.
*   **Economies of Scale:** Certain components are shared or similar across many product lines, allowing Tesla to leverage pricing efficiencies through economies of scale.
*   **Single-Source Risk:** Similar to other OEMs, some procured components and systems are sourced from single suppliers. This creates a vulnerability where production risks are heightened if a disruption occurs.
*   **Mitigation via Multi-Sourcing:** Where multiple sources are available for key components, Tesla works to qualify multiple suppliers to minimize production risks.
*   **Inventory Buffering:** To mitigate supply disruptions, Tesla maintains safety stock for key parts and assemblies, as well as "die banks" for components with lengthy procurement lead times.

### 3. Battery Cell Dependency and Vertical Integration

The supply of lithium-ion battery cells is identified as a critical dependency for both vehicle and energy storage product growth.

*   **Current Suppliers:** Tesla currently relies on suppliers such as **Panasonic** and **Contemporary Amperex Technology Co. Limited (CATL)** for lithium-ion cells.
*   **Limited Flexibility:** The text notes that Tesla has "fully qualified only a very limited number of such suppliers" and has "limited flexibility in changing suppliers."
*   **Vertical Integration Strategy:** To reduce dependency, Tesla intends to supplement supplier cells with in-house manufactured cells. The company believes in-house cells will be more efficient, manufacturable at greater volumes, and more cost-effective than currently available commercial cells.
*   **Investment Requirements:** Developing and manufacturing in-house cells requires significant investments. There is no assurance that these targets will be achieved in the planned timeframes.
*   **Operational Milestone:** An in-house lithium refinery in Texas began operations in **January 2026**, representing a step toward vertical integration and supply chain de-risking.

### 4. Raw Material and Geopolitical Risks

Tesla’s cost structure and supply availability are exposed to fluctuations in raw material markets and geopolitical trade policies.

*   **Key Raw Materials:** Products use aluminum, steel, lithium, nickel, and copper.
*   **Price Volatility:** Pricing for these materials is governed by market conditions and may fluctuate due to supply and demand, market speculation, and refining capacity.
*   **Geopolitical Impact:** U.S. trade policy alterations in **2025**, including heightened import tariffs and retaliatory measures, have already impacted supply chain costs. These policies may further impact the availability of certain technologies or components depending on the final scope of tariffs and export controls.
*   **Supply Constraints:** Increased global production of electric vehicles and energy storage products may result in suppliers being unable to meet Tesla’s volume needs. Additionally, suppliers may face challenges in meeting timelines, cost, or quality requirements, potentially forcing Tesla to replace them.

### 5. Operational and Service Risks

Beyond component sourcing, Tesla faces risks related to the scalability of its service and charging infrastructure.

*   **Servicing Capacity:** Tesla recommends its vehicles be serviced by the company or authorized professionals. Delays in adding servicing capacity or reliability issues with higher-volume models (specifically **Model 3** and **Model Y**) could overburden servicing capabilities and parts inventory.
*   **Charging Infrastructure:** The increasing number of Tesla vehicles requires a rapid increase in Supercharger stations and connectors globally.
*   **New Business Lines:** Expansion into **Robotaxi** and **Semi** businesses requires accurate forecasting and optimization of service and charging capabilities to ensure service quality.
*   **Financial Impact of Disruption:** If Tesla fails to manage growth effectively, it may face cost overruns, inability to hire necessary personnel, and potential harm to brand, business prospects, financial condition, and operating results.

### 6. Overall Supply Chain Vulnerability and Mitigation Assessment

**Vulnerability Assessment:**
Tesla’s supply chain exhibits high vulnerability in two specific areas:
1.  **Battery Cell Supply:** Reliance on a limited number of qualified suppliers (Panasonic, CATL) with limited flexibility to switch creates a single point of failure risk.
2.  **Raw Material Exposure:** Dependence on volatile commodities (lithium, nickel) and exposure to 2025 trade policy changes creates cost and availability uncertainty.

**Mitigation Strategies Identified:**
1.  **Vertical Integration:** Development of in-house battery cell manufacturing and the operation of the Texas lithium refinery (started Jan 2026) aim to reduce external dependency and control costs.
2.  **Multi-Sourcing:** Qualifying multiple suppliers for key components where possible to minimize production risks.
3.  **Inventory Management:** Maintaining safety stock and die banks for long-lead-time components.
4.  **Local Manufacturing:** Establishing facilities in China and Germany to reduce transportation costs and mitigate tariff impacts.
5.  **Long-Term Contracts:** Striving to execute long-term supply contracts for raw materials at competitive pricing when feasible.

**Conclusion:**
While Tesla has implemented significant mitigation strategies, including vertical integration and localized manufacturing, the company acknowledges that there is "no assurance" that in-house cell production targets will be met or that supply disruptions will be fully avoided. The impact of 2025 trade policies and raw material price fluctuations remains a direct risk to profitability and production volumes.

## Sources
- Tesla.pdf | Page 15
- Tesla.pdf | Page 32
- Tesla.pdf | Page 27
- Tesla.pdf | Page 25
- Tesla.pdf | Page 33

In [33]:
query = """Evaluate Tesla's competitive positioning and long-term corporate strategy.

Focus on:
- Competition in electric vehicles (traditional OEMs and new entrants)
- Competitive advantages in AI, software, and energy ecosystem
- Diversification into Robotaxi, Optimus Bots, and Energy
- Capital allocation priorities (R&D, capex, buybacks)
- Risks from new competitors and regulatory changes

Explain each point above in very much depth and with intuition


Conclude with overall strategic strengths, weaknesses, and investor implications."""

response = financial_rag_with_bleu(query = query,company = "Tesla")

display(Markdown(response))



🔍 Company: Tesla | Query: Evaluate Tesla's competitive positioning and long-term corpo...
   Retrieval Quality: 14/17 words matched
⚠️  Corrective RAG triggered — widening search...
📝 Pass 1: Generating Response
   Retrieval Quality: 14/17 words matched
📊 BERTScore (answer vs. retrieved context) — Precision: 0.7562 | Recall: 0.7692 | F1: 0.7626


# Financial Analysis — Tesla

**Financial Analysis: Tesla’s Competitive Positioning and Long-Term Corporate Strategy**

Based on the retrieved sections from Tesla’s financial filings, the following is a structured evaluation of the company’s competitive landscape, strategic diversification, and risk profile. Note that specific financial figures regarding capital allocation (R&D, capex, buybacks) are not disclosed in the provided text; therefore, the analysis focuses on the qualitative strategic and operational factors present in the context.

### 1. Competition in Electric Vehicles (Traditional OEMs and New Entrants)

Tesla operates in a "highly competitive" worldwide automotive market that is expected to become "even more competitive in the future." The competitive dynamic is characterized by:

*   **Broad Market Entry:** A "significant and growing number" of established and new automobile manufacturers, as well as other companies, have entered or announced plans to enter the electric vehicle (EV), hybrid, and self-driving technology markets.
*   **Resource Asymmetry:** Many competitors possess "significantly more or better-established resources" than Tesla for design, development, manufacturing, distribution, and promotion. These competitors may achieve "additional cost efficiencies" due to favorable location and economic environments.
*   **Segment-Specific Competition:**
    *   **Cybertruck:** Competes with other pickup trucks.
    *   **Model S and Model X:** Compete primarily with premium sedans and premium SUVs.
    *   **Model 3 and Model Y:** Compete with small to medium-sized sedans and compact SUVs.
*   **Strategic Implication:** The influx of competitors promotes the development of the EV market by highlighting the attractiveness of EVs relative to internal combustion vehicles. However, increased competition poses a direct risk of "lower vehicle unit sales, price reductions, revenue shortfalls, loss of customers and loss of market share."

### 2. Competitive Advantages in AI, Software, and Energy Ecosystem

Tesla’s strategy relies on proprietary technology and ecosystem integration rather than solely on vehicle hardware.

*   **Intellectual Property & Innovation:** Tesla places a "strong emphasis" on innovative approaches and proprietary designs. The company prioritizes obtaining patents to ensure "freedom to operate" across all products and technologies.
*   **Patent Pledge:** Tesla has irrevocably pledged not to initiate lawsuits against parties infringing its patents for electric vehicles or related equipment, provided they act in good faith. This strategy is designed to "encourage the advancement of a common, rapidly-evolving platform for electric vehicles," benefiting Tesla, other EV makers, and the broader market.
*   **Energy Ecosystem Integration:** In the energy generation business, Tesla competes based on "price" and the "ease by which customers can switch" to its systems. Key differentiators include:
    *   Aesthetics and superior performance of solar panels.
    *   Ease of installation and integration with **Powerwall**.
    *   The environment is described as "increasingly conducive to the adoption of renewable energy systems."
*   **AI and Autonomy:** The next phase of production growth is explicitly linked to "advances in autonomy." Tesla is developing its own battery cells to achieve "high-volume output, lower capital and production costs and longer range," which supports both automotive and AI-enabled product strategies.

### 3. Diversification into Robotaxi, Optimus Bots, and Energy

Tesla is actively diversifying beyond traditional vehicle sales into high-growth, technology-driven sectors.

*   **Robotaxi Business:** Tesla is launching a Robotaxi business, which requires the development and optimization of "dedicated infrastructure." This includes specific capabilities for:
    *   Vehicle cleaning and maintenance.
    *   Charging.
    *   Security.
    *   Teleoperations.
    *   Fleet management.
    *   *Strategic Goal:* To ensure service quality as the business scales.
*   **Optimus Bots:** Growth is dependent on the ability to develop and commercialize "Bots, including Optimus." The context notes this is a "nascent industry that has yet to develop commercially," indicating high uncertainty but potential for future revenue streams.
*   **Energy Generation and Storage:**
    *   **Competition:** Competes with traditional local utility companies, solar energy companies (some of which only install or only finance), and other manufacturers/developers of competing energy technologies.
    *   **Regulatory Environment:** Subject to state and federal regulations. In some jurisdictions, regulators or utilities have "reduced or eliminated the benefit available under net metering," which could impact customer economics.
    *   **Supply Chain:** Tesla is localizing and de-risking supply chains, including through vertical integration such as its "in-house lithium refinery in Texas, which began operations in January 2026."

### 4. Capital Allocation Priorities (R&D, Capex, Buybacks)

*   **Not Disclosed in Retrieved Sections:** The provided context does not contain specific financial figures or explicit statements regarding capital allocation priorities for Research & Development (R&D), Capital Expenditures (Capex), or share buybacks.
*   **Operational Investment Focus:** While specific dollar amounts are absent, the text highlights significant operational investments:
    *   **Manufacturing Capacity:** Focus on "growing and optimizing" manufacturing capacity at Gigafactories, including for newer vehicle models and next-generation platforms.
    *   **Infrastructure Expansion:** Continuous expansion of delivery, servicing, and charging infrastructure (Supercharger network) to meet demands from both Tesla customers and third-party manufacturers adopting NACS.
    *   **Supply Chain Vertical Integration:** Investment in in-house lithium refining and battery cell development to reduce costs and control supply.

### 5. Risks from New Competitors and Regulatory Changes

*   **Competitive Risks:**
    *   **Market Share Erosion:** Increased competition could lead to "price reductions" and "revenue shortfalls."
    *   **Labor Market Competition:** Tesla faces "strong competition" for talent with specialized knowledge in EVs, engineering, and AI. Employees may leave for other employers due to the "very competitive labor market" or negative publicity.
    *   **Competitor Advantages:** Competitors may benefit more from "government and economic incentives" that favor domestic assembly or local suppliers, potentially negatively impacting Tesla’s profitability.
*   **Regulatory Risks:**
    *   **Net Metering:** Reductions or eliminations of net metering benefits in certain jurisdictions could make Tesla’s energy products less attractive and lead to an "increased rate of customer defaults."
    *   **Interconnection Agreements:** Sales of electricity and non-sale equipment leases (e.g., PPAs) have faced "regulatory challenges" in some states.
    *   **Government Incentives:** Changes in government credits, incentives, and policies globally impact customer ownership decisions.

### Overall Strategic Assessment

**Strategic Strengths:**
1.  **Integrated Ecosystem:** Tesla leverages a closed-loop ecosystem spanning vehicles, energy storage (Powerwall), solar, and charging infrastructure (Supercharger), creating switching costs and customer stickiness.
2.  **Vertical Integration:** Control over critical supply chain components, such as in-house lithium refining and battery cell development, aims to secure cost advantages and supply security.
3.  **Open Innovation Strategy:** The patent pledge fosters industry growth, potentially expanding the total addressable market for EVs and supporting Tesla’s brand as a leader in sustainable technology.
4.  **Diversified Growth Engines:** Exposure to high-growth sectors like Robotaxi and Optimus Bots provides potential upside beyond traditional automotive cycles.

**Strategic Weaknesses:**
1.  **Intense Competitive Pressure:** Facing "significantly more or better-established resources" from traditional OEMs and new entrants, with risks of price wars and margin compression.
2.  **Regulatory Dependency:** Revenue in the energy sector is sensitive to net metering policies and government incentives, which are subject to political and regulatory change.
3.  **Execution Risk in Nascent Markets:** The commercialization of Optimus B

## Sources
- Tesla.pdf | Page 21
- Tesla.pdf | Page 33
- Tesla.pdf | Page 15
- Tesla.pdf | Page 28
- Tesla.pdf | Page 20
- Tesla.pdf | Page 55
- Tesla.pdf | Page 57

In [34]:
print(financial_rag._last_context)

---
type: FinancialText
company: Tesla
source_file: Tesla.pdf
page: 21
---

Table of Contents
Energy Generation Systems
The primary competitors to our energy generation business are the traditional local utility companies that supply energy to our
potential customers. We compete with these traditional utility companies primarily based on price and the ease by which customers can
switch to electricity generated by our energy generation systems. We also compete with solar energy companies that provide products
and services similar to ours. Many solar energy companies only install solar energy systems, while others only provide financing for
these installations. We believe we have a significant expansion opportunity with our offerings, including in terms of the aesthetics,
superior performance and ease of installation and integration with Powerwall of our solar panels, and that the environment is
increasingly conducive to the adoption of renewable energy systems. Intellectual Property
We 